# ES-LLM: Layer-wise Fine-Tuning mit Evolutionary Strategies

Dieses Notebook führt dich durch den kompletten Workflow:
1. **Setup** – Repo klonen, Dependencies installieren
2. **Inspect** – Modell-Architektur inspizieren
3. **Baseline** – Unveränderte Accuracy messen
4. **Train** – ES-Training auf ausgewählten Layern
5. **Evaluate** – Fine-tuned Modell testen
6. **Export** – Ergebnisse nach Google Drive speichern

**GPU-Empfehlung:** Runtime → Change runtime type → **T4 GPU** (kostenlos) oder **A100** (Colab Pro)

## 0. GPU prüfen

In [ ]:
import torch
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Keine GPU! Gehe zu Runtime → Change runtime type → GPU")

## 1. Setup – Repo klonen & Dependencies installieren

In [ ]:
# ── Repo klonen ──
import os

REPO_URL = "https://github.com/Haso001/PG_ML_AI.git"
REPO_DIR = "/content/PG_ML_AI"
PROJECT_DIR = f"{REPO_DIR}/es-llm-finetune-Hasan-Dev"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Repo existiert → neueste Änderungen pullen
    !cd {REPO_DIR} && git pull
    print(f"{REPO_DIR} aktualisiert (git pull).")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Dependencies installieren ──
!pip install -q torch transformers datasets accelerate pyyaml cmaes

# Verifizieren
import torch, transformers
print(f"torch={torch.__version__}, transformers={transformers.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# ── src/ zum Python-Pfad hinzufügen ──
import sys
sys.path.insert(0, f"{PROJECT_DIR}/src")

# Import-Test
from es_llm.model.loader import load_model_and_tokenizer, inspect_model_layers
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness
from es_llm.training.es_trainer import train
from es_llm.utils.config import load_config
print("Alle Imports erfolgreich!")

## 2. Baseline messen

Wie gut ist das Modell **unverändert** auf GSM8K und Hellaswag?

In [ ]:
# 4b) OLMES Eval: Baseline auf GSM8K + HellaSwag (offizielles OLMES-Repo, Colab A100)

import json
import subprocess
from datetime import datetime
from pathlib import Path

# ---- Standalone Defaults ----
model_name = globals().get("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
output_base = globals().get("OUTPUT_BASE", Path("/content/experiments/olmes"))
run_tag = globals().get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
if not isinstance(output_base, Path):
    output_base = Path(output_base)

BASE_OLMES_OUT = output_base / f"olmes_baseline_gsm8k_hellaswag_{run_tag}"
BASE_GSM8K_OUT = BASE_OLMES_OUT / "gsm8k"
BASE_HELLASWAG_OUT = BASE_OLMES_OUT / "hellaswag"

for p in [BASE_OLMES_OUT, BASE_GSM8K_OUT, BASE_HELLASWAG_OUT]:
    p.mkdir(parents=True, exist_ok=True)

globals()["BASE_OLMES_OUT"] = BASE_OLMES_OUT
globals()["BASE_GSM8K_OUT"] = BASE_GSM8K_OUT
globals()["BASE_HELLASWAG_OUT"] = BASE_HELLASWAG_OUT

print(f"Model: {model_name}")
print(f"Output root: {BASE_OLMES_OUT}")
print(f"GSM8K out:   {BASE_GSM8K_OUT}")
print(f"HellaSwag out: {BASE_HELLASWAG_OUT}")

def run(cmd, check=True, cwd=None):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, text=True, capture_output=True, cwd=cwd)
    if p.stdout:
        print(p.stdout[-1500:])
    if p.returncode != 0 and p.stderr:
        print(p.stderr[-1500:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
    return p

# ---- 1) Offizielles OLMES klonen ----
OLMES_DIR = Path("/content/olmes")
if not OLMES_DIR.exists():
    run(["git", "clone", "https://github.com/allenai/olmes.git", str(OLMES_DIR)])
else:
    run(["git", "-C", str(OLMES_DIR), "pull"] )

# ---- 2) Installation (pip editable, wie in offizieller Doku) ----
run(["python3", "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"] )
run(["python3", "-m", "pip", "install", "-q", "-e", "."], cwd=str(OLMES_DIR))
gpu_install = run(["python3", "-m", "pip", "install", "-q", "-e", ".[gpu]"], check=False, cwd=str(OLMES_DIR))
if gpu_install.returncode == 0:
    print("OLMES GPU extras installiert (vLLM verfuegbar).")
else:
    print("Hinweis: .[gpu] konnte nicht installiert werden. Nutze --model_type hf (funktioniert auf A100).")

# ---- 3) Baseline-Eval: getrennte Runs je Task ----
script_path = Path("scripts/eval_olmes_tasks.py")
if not script_path.exists():
    raise FileNotFoundError("scripts/eval_olmes_tasks.py nicht gefunden.")

task_runs = [
    ("gsm8k::olmes", BASE_GSM8K_OUT),
    ("hellaswag::olmo1", BASE_HELLASWAG_OUT),
]

for task_name, task_out in task_runs:
    cmd = [
        "python3", str(script_path),
        "--model", model_name,
        "--tasks", task_name,
        "--model_type", "hf",
        "--limit", "200",
        "--num_shots", "4",
        "--batch_size", "8",
        "--cache_dir", "/content/cache",
        "--output_dir", str(task_out),
    ]
    run(cmd, check=True)

summary = {
    "model": model_name,
    "output_root": str(BASE_OLMES_OUT),
    "task_outputs": {
        "gsm8k::olmes": str(BASE_GSM8K_OUT),
        "hellaswag::olmo1": str(BASE_HELLASWAG_OUT),
    },
}
(BASE_OLMES_OUT / "combined_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("✅ Baseline OLMES outputs:")
print("- GSM8K:", BASE_GSM8K_OUT)
print("- HellaSwag:", BASE_HELLASWAG_OUT)

In [ ]:
# 4c) Visualisierung: Baseline GSM8K + HellaSwag (OLMES, standalone)

import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def _read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _extract_task_entries(metrics_obj):
    if isinstance(metrics_obj.get("tasks"), list):
        return metrics_obj["tasks"]
    if isinstance(metrics_obj.get("tasks"), dict):
        out = []
        for name, val in metrics_obj["tasks"].items():
            if isinstance(val, dict):
                e = dict(val)
                e.setdefault("task_name", name)
                out.append(e)
        return out
    return []


def _normalize_task_name(raw_name: str) -> str:
    return (raw_name or "").strip().lower().replace(" ", "")


def _pick_accuracy_metric(task_name: str, row: dict):
    t = _normalize_task_name(task_name)
    if "gsm8k" in t:
        for key in ["exact_match", "exact_match_flex", "primary_score", "acc_norm", "acc_raw"]:
            val = row.get(key)
            if val is not None and pd.notna(val):
                return key, float(val)
    if "hellaswag" in t:
        for key in ["acc_norm", "acc_raw", "primary_score", "exact_match", "exact_match_flex"]:
            val = row.get(key)
            if val is not None and pd.notna(val):
                return key, float(val)
    for key in ["primary_score", "acc_norm", "acc_raw", "exact_match", "exact_match_flex"]:
        val = row.get(key)
        if val is not None and pd.notna(val):
            return key, float(val)
    return None, None


def _resolve_output_root() -> Path:
    root = globals().get("BASE_OLMES_OUT")
    if root is not None and (Path(root) / "combined_summary.json").exists():
        return Path(root)

    base = Path(globals().get("OUTPUT_BASE", Path.cwd() / "experiments" / "olmes"))
    candidates = sorted(base.glob("olmes_baseline_gsm8k_hellaswag_*"), key=lambda p: p.stat().st_mtime, reverse=True)
    for c in candidates:
        if (c / "combined_summary.json").exists():
            globals()["BASE_OLMES_OUT"] = c
            return c
    raise FileNotFoundError("Kein combined_summary.json gefunden. Fuehre zuerst Zelle 19 aus.")


def _resolve_task_output(root: Path, task_key: str, fallback_subdir: str) -> Path:
    summary_path = root / "combined_summary.json"
    if summary_path.exists():
        summary = _read_json(summary_path)
        task_outputs = summary.get("task_outputs", {})
        p = task_outputs.get(task_key)
        if p and (Path(p) / "metrics.json").exists():
            return Path(p)

    fallback = root / fallback_subdir
    if (fallback / "metrics.json").exists():
        return fallback
    raise FileNotFoundError(f"metrics.json fuer {task_key} nicht gefunden unter {fallback}")


def _task_row_from_metrics(metrics_path: Path, expected_task: str):
    metrics = _read_json(metrics_path)
    entries = _extract_task_entries(metrics)

    if not entries:
        return {
            "task": expected_task,
            "task_norm": _normalize_task_name(expected_task),
            "accuracy_metric": None,
            "accuracy": None,
            "source": str(metrics_path),
        }

    # Prefer entry matching expected task; otherwise take first entry.
    chosen = None
    for e in entries:
        name = e.get("alias") or e.get("task_name") or e.get("name") or e.get("id") or "unknown_task"
        if expected_task.split("::")[0] in _normalize_task_name(name):
            chosen = e
            break
    if chosen is None:
        chosen = entries[0]

    m = chosen.get("metrics", {}) or {}
    task_name = chosen.get("alias") or chosen.get("task_name") or chosen.get("name") or chosen.get("id") or expected_task
    row = {
        "task": task_name,
        "task_norm": _normalize_task_name(task_name),
        "primary_score": m.get("primary_score", chosen.get("primary_score")),
        "acc_raw": m.get("acc_raw", chosen.get("acc_raw")),
        "acc_norm": m.get("acc_norm", chosen.get("acc_norm")),
        "exact_match": m.get("exact_match", chosen.get("exact_match")),
        "exact_match_flex": m.get("exact_match_flex", chosen.get("exact_match_flex")),
        "num_instances": chosen.get("num_instances"),
        "processing_time_sec": chosen.get("processing_time"),
        "source": str(metrics_path),
    }
    metric_name, metric_val = _pick_accuracy_metric(task_name, row)
    row["accuracy_metric"] = metric_name
    row["accuracy"] = metric_val
    return row


root = _resolve_output_root()
gsm_metrics = _resolve_task_output(root, "gsm8k::olmes", "gsm8k") / "metrics.json"
hsw_metrics = _resolve_task_output(root, "hellaswag::olmo1", "hellaswag") / "metrics.json"

rows = [
    _task_row_from_metrics(gsm_metrics, "gsm8k::olmes"),
    _task_row_from_metrics(hsw_metrics, "hellaswag::olmo1"),
]
baseline_df = pd.DataFrame(rows)
display(baseline_df)

plot_df = baseline_df.dropna(subset=["accuracy"]).copy()
if plot_df.empty:
    print("Keine plotbaren Accuracy-Werte gefunden.")
else:
    plt.figure(figsize=(9, 4))
    plt.bar(plot_df["task"], plot_df["accuracy"])
    plt.title("Baseline OLMES: GSM8K vs HellaSwag (Accuracy)")
    plt.ylabel("accuracy")
    plt.xlabel("task")
    plt.ylim(0, 1)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

gsm = baseline_df[baseline_df["task_norm"].str.contains("gsm8k", na=False)]
hsw = baseline_df[baseline_df["task_norm"].str.contains("hellaswag", na=False)]

if not gsm.empty and not hsw.empty and pd.notna(gsm.iloc[0]["accuracy"]) and pd.notna(hsw.iloc[0]["accuracy"]):
    gsm_row = gsm.iloc[0]
    hsw_row = hsw.iloc[0]
    print("\nAccuracy-Vergleich (Baseline):")
    print(f"- GSM8K ({gsm_row['accuracy_metric']}): {gsm_row['accuracy']:.4f}")
    print(f"- HellaSwag ({hsw_row['accuracy_metric']}): {hsw_row['accuracy']:.4f}")
    print(f"- Differenz (HellaSwag - GSM8K): {hsw_row['accuracy'] - gsm_row['accuracy']:+.4f}")
else:
    print("Vergleich unvollstaendig: Mindestens ein Task ohne valide Accuracy.")
    print("GSM8K metrics:", gsm_metrics)
    print("HellaSwag metrics:", hsw_metrics)

print("Baseline OLMES output root:", root)
print("- GSM8K metrics:", gsm_metrics)
print("- HellaSwag metrics:", hsw_metrics)

## 3. ES-Training starten



### 3.0 Binary-Validator ES (Layer 20+21, 30 Generationen, <=3h Ziel)

Ziel dieses Blocks:
- **0/1-Fitness** auf GSM8K (strict numeric match)
- Training nur auf **Layer [20, 21]**
- **30 Generationen** mit laufender Fortschrittsanzeige
- Danach: **Vergleich Fine-Tuned vs Base** + Visualisierung

In [ ]:
# 3.0.1 Training: Binary-Validator auf GSM8K mit Layer [20, 21]

import random
import re
import time
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k import GSM8KFitness


SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = globals().get("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16" if DEVICE.startswith("cuda") else "float32"

# Laufzeit-sichere Defaults fuer Colab A100 (typisch <= 3h)
BINARY_LAYER_INDICES = [20, 21]
NUM_GENERATIONS = 30
POPULATION_SIZE = 8
SIGMA = 0.001
LEARNING_RATE = 0.001
NUM_SAMPLES_TRAIN = 20
MAX_NEW_TOKENS = 128

global_output_base = Path(globals().get("OUTPUT_BASE", "/content/experiments"))
run_tag = time.strftime("%Y%m%d_%H%M%S")
binary_run_dir = global_output_base / f"binary_l20_l21_g30_{run_tag}"
binary_run_dir.mkdir(parents=True, exist_ok=True)


def _extract_numeric_answer(text: str):
    if text is None:
        return None
    t = text.strip().replace(",", "")
    matches = re.findall(r"-?\d+(?:\.\d+)?", t)
    if not matches:
        return None
    return matches[-1]


print("Lade Modell + Tokenizer ...")
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    device=DEVICE,
    dtype=DTYPE,
)
model.eval()

selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=BINARY_LAYER_INDICES,
    components="attention",
)

if selector.num_target_elements == 0:
    raise RuntimeError(
        "LayerSelector hat 0 Parameter selektiert. Pruefe layer_indices/components."
    )

print(f"Trainierbare Parameter: {len(selector.target_names)} tensors, {selector.num_target_elements:,} params")

es = OpenAIES(
    population_size=POPULATION_SIZE,
    sigma=SIGMA,
    learning_rate=LEARNING_RATE,
    antithetic=True,
    optimizer="adam",
    fitness_shaping="centered_rank",
    seed=SEED,
    sigma_decay="constant",
    max_generations=NUM_GENERATIONS,
)

fitness_fn = GSM8KFitness(
    num_samples=NUM_SAMPLES_TRAIN,
    split="train",
    max_new_tokens=MAX_NEW_TOKENS,
    seed=SEED,
    reshuffle=True,
    pool_size=4000,
)

current_params = selector.get_flat_params().clone()
best_params = current_params.clone()
best_ever = -float("inf")
history_rows = []

print("\nStarte ES-Training ...")
print(f"Konfig: gens={NUM_GENERATIONS}, pop={POPULATION_SIZE}, samples={NUM_SAMPLES_TRAIN}, sigma={SIGMA}, lr={LEARNING_RATE}")

train_start = time.time()
for gen in range(1, NUM_GENERATIONS + 1):
    gen_start = time.time()

    fitness_fn.reshuffle_data()
    candidates = es.ask(current_params)
    fitness_values = []

    for cand in tqdm(candidates, desc=f"Gen {gen}/{NUM_GENERATIONS}", leave=False):
        selector.set_flat_params(cand)
        fit = float(fitness_fn.evaluate(model, tokenizer))
        fitness_values.append(fit)

    result = es.tell(current_params, candidates, fitness_values)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)

    gen_best = float(result.best_fitness)
    gen_mean = float(result.mean_fitness)
    gen_std = float(result.std_fitness)

    if gen_best > best_ever:
        best_ever = gen_best
        best_params = current_params.clone()

    gen_sec = time.time() - gen_start
    elapsed_min = (time.time() - train_start) / 60.0
    eta_min = (elapsed_min / gen) * (NUM_GENERATIONS - gen)

    history_rows.append(
        {
            "generation": gen,
            "best": gen_best,
            "mean": gen_mean,
            "std": gen_std,
            "best_ever": best_ever,
            "sigma": float(es.current_sigma),
            "gen_seconds": gen_sec,
            "elapsed_minutes": elapsed_min,
            "eta_minutes": eta_min,
        }
    )

    print(
        f"[Gen {gen:02d}/{NUM_GENERATIONS}] best={gen_best:.3f} mean={gen_mean:.3f} "
        f"best_ever={best_ever:.3f} sigma={es.current_sigma:.5f} "
        f"time={gen_sec:.1f}s elapsed={elapsed_min:.1f}m eta={eta_min:.1f}m"
    )

# Beste Gewichte ins Modell schreiben und speichern
selector.set_flat_params(best_params)

binary_history_df = pd.DataFrame(history_rows)
binary_history_path = binary_run_dir / "binary_training_history.csv"
binary_history_df.to_csv(binary_history_path, index=False)

binary_model_dir = binary_run_dir / "best_model"
binary_model_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(binary_model_dir)
tokenizer.save_pretrained(binary_model_dir)

print("\nTraining abgeschlossen.")
print("Run dir:", binary_run_dir)
print("History:", binary_history_path)
print("Best model:", binary_model_dir)
print("Best-ever fitness:", best_ever)

# Fuer Folgezellen bereitstellen
globals()["binary_run_dir"] = str(binary_run_dir)
globals()["binary_model_dir"] = str(binary_model_dir)
globals()["binary_history_df"] = binary_history_df


In [ ]:
# 3.0.2 Evaluation vs Base + Visualisierung

import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import load_dataset

from es_llm.model.loader import load_model_and_tokenizer


if "binary_model_dir" not in globals():
    raise RuntimeError("Bitte zuerst die Trainingszelle 3.0.1 ausfuehren.")

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = globals().get("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16" if DEVICE.startswith("cuda") else "float32"

EVAL_SAMPLES = 200
MAX_NEW_TOKENS_EVAL = 96


def eval_binary_accuracy(model, tokenizer, dataset, n_samples=200, seed=42, max_new_tokens=96):
    rng = random.Random(seed)
    n_samples = min(n_samples, len(dataset))
    indices = rng.sample(range(len(dataset)), k=n_samples)
    correct = 0

    for idx in indices:
        ex = dataset[int(idx)]
        prompt = (
            "Solve the following math word problem. "
            "Think step by step and end with 'Final answer: <number>'.\n\n"
            f"Question: {ex['question']}\nAnswer:"
        )
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = generated[0, inputs["input_ids"].shape[1]:]
        pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

        pred_num = _extract_numeric_answer(pred_text)
        gold_num = _extract_numeric_answer(ex["answer"])
        if pred_num is not None and gold_num is not None and pred_num == gold_num:
            correct += 1

    acc = correct / float(n_samples)
    return {"accuracy": acc, "correct": correct, "total": n_samples}


print("Lade GSM8K test split ...")
gsm8k_test = load_dataset("gsm8k", "main", split="test")

print("Lade Base-Modell ...")
base_model, base_tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    device=DEVICE,
    dtype=DTYPE,
)
base_model.eval()

print("Lade Fine-Tuned Modell ...")
ft_model, ft_tokenizer = load_model_and_tokenizer(
    model_name=binary_model_dir,
    device=DEVICE,
    dtype=DTYPE,
)
ft_model.eval()

print("\nEval Base ...")
t0 = time.time()
base_res = eval_binary_accuracy(
    base_model,
    base_tokenizer,
    gsm8k_test,
    n_samples=EVAL_SAMPLES,
    seed=SEED,
    max_new_tokens=MAX_NEW_TOKENS_EVAL,
)
print(f"Base accuracy: {base_res['accuracy']:.4f} ({base_res['correct']}/{base_res['total']}) in {(time.time()-t0):.1f}s")

print("Eval Fine-Tuned ...")
t1 = time.time()
ft_res = eval_binary_accuracy(
    ft_model,
    ft_tokenizer,
    gsm8k_test,
    n_samples=EVAL_SAMPLES,
    seed=SEED,
    max_new_tokens=MAX_NEW_TOKENS_EVAL,
)
print(f"Fine-tuned accuracy: {ft_res['accuracy']:.4f} ({ft_res['correct']}/{ft_res['total']}) in {(time.time()-t1):.1f}s")

acc_delta = ft_res["accuracy"] - base_res["accuracy"]
print(f"Delta (FT - Base): {acc_delta:+.4f}")

# Ergebnis sichern
run_dir = Path(binary_run_dir)
eval_payload = {
    "model_name": MODEL_NAME,
    "binary_model_dir": binary_model_dir,
    "eval_samples": EVAL_SAMPLES,
    "max_new_tokens": MAX_NEW_TOKENS_EVAL,
    "base": base_res,
    "fine_tuned": ft_res,
    "delta": acc_delta,
}

eval_path = run_dir / "eval_vs_base.json"
with eval_path.open("w", encoding="utf-8") as f:
    json.dump(eval_payload, f, indent=2)
print("Eval saved:", eval_path)

# Visualisierung
hist_df = globals().get("binary_history_df", None)
if hist_df is None or len(hist_df) == 0:
    hist_csv = run_dir / "binary_training_history.csv"
    if hist_csv.exists():
        hist_df = pd.read_csv(hist_csv)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if hist_df is not None and len(hist_df) > 0:
    axes[0].plot(hist_df["generation"], hist_df["best"], label="best", linewidth=2)
    axes[0].plot(hist_df["generation"], hist_df["mean"], label="mean", linewidth=2)
    axes[0].plot(hist_df["generation"], hist_df["best_ever"], label="best_ever", linewidth=2, linestyle="--")
    axes[0].set_title("Binary ES Training-Kurve")
    axes[0].set_xlabel("Generation")
    axes[0].set_ylabel("Fitness (Accuracy)")
    axes[0].grid(alpha=0.3)
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "Keine History verfuegbar", ha="center", va="center")
    axes[0].set_axis_off()

bars = [base_res["accuracy"], ft_res["accuracy"]]
labels = ["Base", "Fine-Tuned"]
colors = ["#7a7a7a", "#2a9d8f"]
axes[1].bar(labels, bars, color=colors)
axes[1].set_ylim(0, max(0.1, min(1.0, max(bars) + 0.08)))
axes[1].set_title(f"GSM8K Binary Accuracy (n={EVAL_SAMPLES})")
axes[1].set_ylabel("Accuracy")
for i, v in enumerate(bars):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha="center")

plt.suptitle(f"Base vs Fine-Tuned | Delta: {acc_delta:+.4f}")
plt.tight_layout()
plt.show()


In [ ]:
# 3.0.x Baseline-Vergleich: HellaSwag-Accuracy ohne Finetuning

from es_llm.fitness.hellaswag import HellaSwagBinaryFitness
from es_llm.model.loader import load_model_and_tokenizer

MODEL_NAME = globals().get("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16" if DEVICE.startswith("cuda") else "float32"

print("Lade Modell + Tokenizer ...")
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    device=DEVICE,
    dtype=DTYPE,
)
model.eval()

fitness_fn = HellaSwagBinaryFitness(
    num_samples=100,
    split="validation",
    max_new_tokens=32,
    seed=42,
    reshuffle=True,
    pool_size=4000,
)

acc = fitness_fn.evaluate(model, tokenizer)
print(f"Baseline-Accuracy auf HellaSwag (n=100): {acc:.3f}")
globals()["hellaswag_baseline_acc"] = acc


In [ ]:
# 3.0.x Training: Binary-Validator auf HellaSwag mit Layer [20, 21]

import random
import time
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.hellaswag import HellaSwagBinaryFitness

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = globals().get("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16" if DEVICE.startswith("cuda") else "float32"

BINARY_LAYER_INDICES = [20, 21]
NUM_GENERATIONS = 30
POPULATION_SIZE = 8
SIGMA = 0.001
LEARNING_RATE = 0.001
NUM_SAMPLES_TRAIN = 20
MAX_NEW_TOKENS = 32

global_output_base = Path(globals().get("OUTPUT_BASE", "/content/experiments"))
run_tag = time.strftime("%Y%m%d_%H%M%S")
hellaswag_run_dir = global_output_base / f"hellaswag_l20_l21_g30_{run_tag}"
hellaswag_run_dir.mkdir(parents=True, exist_ok=True)

print("Lade Modell + Tokenizer ...")
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    device=DEVICE,
    dtype=DTYPE,
)
model.eval()

selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=BINARY_LAYER_INDICES,
    components="attention",
)

if selector.num_target_elements == 0:
    raise RuntimeError(
        "LayerSelector hat 0 Parameter selektiert. Pruefe layer_indices/components."
    )

print(f"Trainierbare Parameter: {len(selector.target_names)} tensors, {selector.num_target_elements:,} params")

es = OpenAIES(
    population_size=POPULATION_SIZE,
    sigma=SIGMA,
    learning_rate=LEARNING_RATE,
    antithetic=True,
    optimizer="adam",
    fitness_shaping="centered_rank",
    seed=SEED,
    sigma_decay="constant",
    max_generations=NUM_GENERATIONS,
)

fitness_fn = HellaSwagBinaryFitness(
    num_samples=NUM_SAMPLES_TRAIN,
    split="validation",
    max_new_tokens=MAX_NEW_TOKENS,
    seed=SEED,
    reshuffle=True,
    pool_size=4000,
)

current_params = selector.get_flat_params().clone()
best_params = current_params.clone()
best_ever = -float("inf")
history_rows = []

print("\nStarte ES-Training auf HellaSwag ...")
print(f"Konfig: gens={NUM_GENERATIONS}, pop={POPULATION_SIZE}, samples={NUM_SAMPLES_TRAIN}, sigma={SIGMA}, lr={LEARNING_RATE}")

train_start = time.time()
for gen in range(1, NUM_GENERATIONS + 1):
    gen_start = time.time()

    fitness_fn.reshuffle_data()
    candidates = es.ask(current_params)
    fitness_values = []

    for cand in tqdm(candidates, desc=f"Gen {gen}/{NUM_GENERATIONS}", leave=False):
        selector.set_flat_params(cand)
        fit = float(fitness_fn.evaluate(model, tokenizer))
        fitness_values.append(fit)

    result = es.tell(current_params, candidates, fitness_values)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)

    gen_best = float(result.best_fitness)
    gen_mean = float(result.mean_fitness)
    gen_std = float(result.std_fitness)

    if gen_best > best_ever:
        best_ever = gen_best
        best_params = current_params.clone()

    gen_sec = time.time() - gen_start
    elapsed_min = (time.time() - train_start) / 60.0
    eta_min = (elapsed_min / gen) * (NUM_GENERATIONS - gen)

    history_rows.append(
        {
            "generation": gen,
            "best": gen_best,
            "mean": gen_mean,
            "std": gen_std,
            "best_ever": best_ever,
            "sigma": float(es.current_sigma),
            "gen_seconds": gen_sec,
            "elapsed_minutes": elapsed_min,
            "eta_minutes": eta_min,
        }
    )

    print(
        f"[Gen {gen:02d}/{NUM_GENERATIONS}] best={gen_best:.3f} mean={gen_mean:.3f} "
        f"best_ever={best_ever:.3f} sigma={es.current_sigma:.5f} "
        f"time={gen_sec:.1f}s elapsed={elapsed_min:.1f}m eta={eta_min:.1f}m"
    )

# Beste Gewichte ins Modell schreiben und speichern
selector.set_flat_params(best_params)

hellaswag_history_df = pd.DataFrame(history_rows)
hellaswag_history_path = hellaswag_run_dir / "hellaswag_training_history.csv"
hellaswag_history_df.to_csv(hellaswag_history_path, index=False)

hellaswag_model_dir = hellaswag_run_dir / "best_model"
hellaswag_model_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(hellaswag_model_dir)
tokenizer.save_pretrained(hellaswag_model_dir)

print("\nTraining abgeschlossen.")
print("Run dir:", hellaswag_run_dir)
print("History:", hellaswag_history_path)
print("Best model:", hellaswag_model_dir)
print("Best-ever fitness:", best_ever)

globals()["hellaswag_run_dir"] = str(hellaswag_run_dir)
globals()["hellaswag_model_dir"] = str(hellaswag_model_dir)
globals()["hellaswag_history_df"] = hellaswag_history_df


In [ ]:
# 3.0.x Evaluation vs Base + Visualisierung (HellaSwag, gleiches Dataset wie Training)

import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import load_dataset

from es_llm.model.loader import load_model_and_tokenizer


if "binary_model_dir" not in globals():
    raise RuntimeError("Bitte zuerst die HellaSwag-Trainingszelle 3.0.x ausfuehren.")

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = globals().get("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16" if DEVICE.startswith("cuda") else "float32"

EVAL_SAMPLES = 200
EVAL_SPLIT = "validation"   # fuer Rowan/hellaswag sinnvoller Eval-Split


def avg_logprob_for_suffix(model, tokenizer, prompt: str, suffix: str) -> float:
    full_text = prompt + suffix

    tok_full = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )
    tok_prompt = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )

    tok_full = {k: v.to(model.device) for k, v in tok_full.items()}
    tok_prompt = {k: v.to(model.device) for k, v in tok_prompt.items()}

    with torch.no_grad():
        out = model(**tok_full)
        logits = out.logits[:, :-1, :]
        labels = tok_full["input_ids"][:, 1:]
        log_probs = torch.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    prompt_len = tok_prompt["input_ids"].shape[1] - 1
    suffix_log_probs = token_log_probs[:, prompt_len:]

    if suffix_log_probs.numel() == 0:
        return -1e9

    return float(suffix_log_probs.mean().item())


def eval_hellaswag_accuracy(model, tokenizer, dataset, n_samples=200, seed=42):
    rng = random.Random(seed)
    n_samples = min(n_samples, len(dataset))
    indices = rng.sample(range(len(dataset)), k=n_samples)

    correct = 0

    for idx in indices:
        ex = dataset[int(idx)]
        ctx = ex.get("ctx") or ""
        endings = ex.get("endings") or []

        if not endings:
            continue

        prompt = (
            "Choose the most plausible continuation for the context.\n"
            f"Context: {ctx}\n"
            "Continuation: "
        )

        scores = [
            avg_logprob_for_suffix(model, tokenizer, prompt, ending)
            for ending in endings
        ]

        pred_idx = int(max(range(len(scores)), key=lambda i: scores[i]))
        gold_idx = int(ex["label"])

        if pred_idx == gold_idx:
            correct += 1

    acc = correct / float(n_samples)
    return {"accuracy": acc, "correct": correct, "total": n_samples}


print(f"Lade HellaSwag split: {EVAL_SPLIT} ...")
hellaswag_eval = load_dataset("Rowan/hellaswag", split=EVAL_SPLIT)

print("Lade Base-Modell ...")
base_model, base_tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    device=DEVICE,
    dtype=DTYPE,
)
base_model.eval()

print("Lade Fine-Tuned Modell ...")
ft_model, ft_tokenizer = load_model_and_tokenizer(
    model_name=binary_model_dir,
    device=DEVICE,
    dtype=DTYPE,
)
ft_model.eval()

print("\nEval Base ...")
t0 = time.time()
base_res = eval_hellaswag_accuracy(
    base_model,
    base_tokenizer,
    hellaswag_eval,
    n_samples=EVAL_SAMPLES,
    seed=SEED,
)
print(
    f"Base accuracy: {base_res['accuracy']:.4f} "
    f"({base_res['correct']}/{base_res['total']}) in {(time.time()-t0):.1f}s"
)

print("Eval Fine-Tuned ...")
t1 = time.time()
ft_res = eval_hellaswag_accuracy(
    ft_model,
    ft_tokenizer,
    hellaswag_eval,
    n_samples=EVAL_SAMPLES,
    seed=SEED,
)
print(
    f"Fine-tuned accuracy: {ft_res['accuracy']:.4f} "
    f"({ft_res['correct']}/{ft_res['total']}) in {(time.time()-t1):.1f}s"
)

acc_delta = ft_res["accuracy"] - base_res["accuracy"]
print(f"Delta (FT - Base): {acc_delta:+.4f}")

# Ergebnis sichern
run_dir = Path(binary_run_dir)
eval_payload = {
    "model_name": MODEL_NAME,
    "binary_model_dir": binary_model_dir,
    "dataset": "Rowan/hellaswag",
    "split": EVAL_SPLIT,
    "eval_samples": EVAL_SAMPLES,
    "base": base_res,
    "fine_tuned": ft_res,
    "delta": acc_delta,
}

eval_path = run_dir / "hellaswag_eval_vs_base.json"
with eval_path.open("w", encoding="utf-8") as f:
    json.dump(eval_payload, f, indent=2)

print("Eval saved:", eval_path)

# Visualisierung
hist_df = globals().get("binary_history_df", None)
if hist_df is None or len(hist_df) == 0:
    hist_csv = run_dir / "binary_training_history.csv"
    if hist_csv.exists():
        hist_df = pd.read_csv(hist_csv)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if hist_df is not None and len(hist_df) > 0:
    gen_col = "generation" if "generation" in hist_df.columns else "gen"

    axes[0].plot(hist_df[gen_col], hist_df["best"], label="best", linewidth=2)
    axes[0].plot(hist_df[gen_col], hist_df["mean"], label="mean", linewidth=2)

    if "best_ever" in hist_df.columns:
        axes[0].plot(
            hist_df[gen_col],
            hist_df["best_ever"],
            label="best_ever",
            linewidth=2,
            linestyle="--",
        )

    axes[0].set_title("Binary ES Training-Kurve (HellaSwag)")
    axes[0].set_xlabel("Generation")
    axes[0].set_ylabel("Fitness (Accuracy)")
    axes[0].grid(alpha=0.3)
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "Keine History verfuegbar", ha="center", va="center")
    axes[0].set_axis_off()

bars = [base_res["accuracy"], ft_res["accuracy"]]
labels = ["Base", "Fine-Tuned"]
colors = ["#7a7a7a", "#2a9d8f"]

axes[1].bar(labels, bars, color=colors)
axes[1].set_ylim(0, max(0.1, min(1.0, max(bars) + 0.08)))
axes[1].set_title(f"HellaSwag Accuracy ({EVAL_SPLIT}, n={EVAL_SAMPLES})")
axes[1].set_ylabel("Accuracy")

for i, v in enumerate(bars):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha="center")

plt.suptitle(f"Base vs Fine-Tuned (HellaSwag) | Delta: {acc_delta:+.4f}")
plt.tight_layout()
plt.show()

### 3.1 Zusatz-Training: Zielgerichtetes ES-Training (0/1-Fitness) fuer **Layer 20+21 / attention**

Dieser zusaetzliche Block fuehrt einen fokussierten ES-Run aus:
- Modell: `Qwen/Qwen2.5-0.5B-Instruct`
- Trainierbare Parameter: **Layer 20 + 21**, nur **attention**
- Fitness: **GSM8K 0/1 Accuracy**

Danach wird das resultierende Modell wie in den anderen Trainings mit dem OLMES-Framework evaluiert.



In [ ]:
# 3.2 Shared Setup: fokussierter 0/1-Run fuer Layer 20/21 attention

import json
import random
import time
import inspect
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import torch

from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k import GSM8KFitness

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = globals().get("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16" if DEVICE.startswith("cuda") else "float32"

RUN_TAG = globals().get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
OUTPUT_BASE = Path(globals().get("OUTPUT_BASE", "/content/experiments"))
TRAIN_ROOT = OUTPUT_BASE / f"es_binary_training_l20_l21_attention_{RUN_TAG}"
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
IS_A100 = "A100" in GPU_NAME.upper()

# Runtime-Profile fuer praktikable Laufzeit in Colab
PROFILE = {
    "a100": {
        "population_size": 40,
        "num_generations": 40,
        "fitness_num_samples": 64,
        "fitness_batch_size": 8,
    },
    "t4": {
        "population_size": 20,
        "num_generations": 36,
        "fitness_num_samples": 40,
        "fitness_batch_size": 4,
    },
}
ACTIVE_PROFILE = PROFILE["a100" if IS_A100 else "t4"]

ES_PARAMS = {
    "population_size": ACTIVE_PROFILE["population_size"],
    "sigma": 0.001,
    "learning_rate": 0.0008,
    "antithetic": True,
    "fitness_shaping": "centered_rank",
    "weight_decay": 0.0,
    "seed": SEED,
}
NUM_GENERATIONS = ACTIVE_PROFILE["num_generations"]
FITNESS_NUM_SAMPLES = ACTIVE_PROFILE["fitness_num_samples"]
FITNESS_BATCH_SIZE = ACTIVE_PROFILE["fitness_batch_size"]
MAX_NEW_TOKENS = 128

TARGET_LAYER_CONFIG = {
    "name": "gsm8k__l20_l21_attention",
    "train_task": "gsm8k",
    "layer_indices": [20, 21],
    "components": "attention",
    "expected_params": "~3.6M",
}

print("Aktives Profil:")
print(json.dumps({
    "gpu": GPU_NAME,
    "is_a100": IS_A100,
    "es_params": ES_PARAMS,
    "num_generations": NUM_GENERATIONS,
    "fitness_num_samples": FITNESS_NUM_SAMPLES,
    "fitness_batch_size": FITNESS_BATCH_SIZE,
}, indent=2))

def build_gsm8k_fitness_compat(**kwargs):
    """Kompatibel mit alter und neuer GSM8KFitness-Signatur."""
    try:
        params = inspect.signature(GSM8KFitness.__init__).parameters
        if "batch_size" not in params and "batch_size" in kwargs:
            kwargs = {k: v for k, v in kwargs.items() if k != "batch_size"}
    except Exception:
        # Fallback: versuchen ohne batch_size
        if "batch_size" in kwargs:
            kwargs = {k: v for k, v in kwargs.items() if k != "batch_size"}
    return GSM8KFitness(**kwargs)



# Kandidaten-Parallelisierung: standardmaessig 10 Worker.
CANDIDATE_EVAL_WORKERS = int(globals().get("CANDIDATE_EVAL_WORKERS", 10))
CANDIDATE_EVAL_WORKERS = max(1, CANDIDATE_EVAL_WORKERS)
print(f"Candidate eval workers: {CANDIDATE_EVAL_WORKERS}")




In [ ]:
# 3.2b Colab-Hotfix: robuster Repair fuer transformers/audioflamingo3/torchvision::nms

import importlib
import os
import subprocess
import sys

# Originalfunktion sichern und mit Auto-Repair wrappen.
_original_load_model_and_tokenizer = load_model_and_tokenizer

# Vision-Imports in transformers fuer dieses Notebook vermeiden (wir machen nur Text-LM).
os.environ.setdefault("TRANSFORMERS_NO_TORCHVISION", "1")


def _repair_transformers_stack():
    """Repariert den HF/transformers-Stack fuer Colab deterministisch.

    Strategie:
    1) Problematische Mischinstallation entfernen (inkl. torchvision)
    2) Auf stabile Versionen pinnen
    3) Module entladen und Loader neu importieren
    """
    print("\n[repair] Entferne inkonsistente HF-Pakete inkl. torchvision ...")
    uninstall = [
        sys.executable, "-m", "pip", "uninstall", "-y",
        "transformers", "tokenizers", "accelerate", "huggingface_hub", "safetensors", "torchvision",
    ]
    subprocess.run(uninstall, check=False)

    print("[repair] Installiere stabile Versionen ...")
    install = [
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade", "--force-reinstall",
        "transformers==4.46.3",
        "tokenizers==0.20.3",
        "accelerate==1.1.1",
        "huggingface_hub==0.26.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call(install)

    # Geladene Module entfernen, damit Python frisch von Disk importiert.
    to_drop = [
        m for m in list(sys.modules.keys())
        if m.startswith("transformers")
        or m.startswith("tokenizers")
        or m.startswith("accelerate")
        or m.startswith("huggingface_hub")
        or m.startswith("torchvision")
    ]
    for m in to_drop:
        sys.modules.pop(m, None)

    # Loader-Modul neu importieren.
    sys.modules.pop("es_llm.model.loader", None)
    loader_mod = importlib.import_module("es_llm.model.loader")
    print("[repair] Reimport erfolgreich.")
    return loader_mod.load_model_and_tokenizer


def _is_known_hf_stack_error(msg: str) -> bool:
    needles = [
        "transformers.models.audioflamingo3",
        "operator torchvision::nms does not exist",
        "Could not import module 'Qwen2ForCausalLM'",
        "requirements defined correctly",
    ]
    m = (msg or "").lower()
    return any(n.lower() in m for n in needles)


def load_model_and_tokenizer(*args, **kwargs):
    try:
        return _original_load_model_and_tokenizer(*args, **kwargs)
    except Exception as e:
        msg = str(e)
        if not _is_known_hf_stack_error(msg):
            raise

        print("\n[warn] Gefundener Colab/HF-Stack Fehler:", msg)
        print("[warn] Versuche automatische Reparatur und Retry...")
        repaired_loader = _repair_transformers_stack()

        try:
            return repaired_loader(*args, **kwargs)
        except Exception as e2:
            raise RuntimeError(
                "Auto-Repair fehlgeschlagen. Bitte Runtime neu starten und dann ab Zelle 12 erneut ausfuehren. "
                f"Letzter Fehler: {e2}"
            ) from e2


print("Robuster Auto-repair Wrapper aktiv: load_model_and_tokenizer")

In [ ]:
# 3.3 Training ausfuehren: gezielt auf Layer 20/21 attention (GSM8K)

def build_train_fitness(reshuffle: bool = True):
    return build_gsm8k_fitness_compat(
        num_samples=FITNESS_NUM_SAMPLES,
        split="train",
        max_new_tokens=MAX_NEW_TOKENS,
        seed=SEED,
        reshuffle=reshuffle,
        pool_size=5000,
        batch_size=FITNESS_BATCH_SIZE,
    )


def _prepare_shared_subset(base_fitness):
    """Fixiert pro Generation ein gemeinsames Subset fuer alle Worker."""
    if hasattr(base_fitness, "reshuffle_data"):
        base_fitness.reshuffle_data()
    if hasattr(base_fitness, "_load_data"):
        base_fitness._load_data()
    return getattr(base_fitness, "_dataset", None)


def _build_worker_bundle(worker_id: int):
    """Laedt eigenes Modell/Selector/Fitness pro Worker."""
    worker_model, worker_tokenizer = load_model_and_tokenizer(
        model_name=MODEL_NAME,
        dtype=DTYPE,
        device=DEVICE,
    )
    worker_selector = LayerSelector(
        model=worker_model,
        strategy="by_index",
        layer_indices=TARGET_LAYER_CONFIG["layer_indices"],
        components=TARGET_LAYER_CONFIG["components"],
    )
    worker_fitness = build_train_fitness(reshuffle=False)
    print(f"  Worker {worker_id}: params={worker_selector.num_target_elements:,}")
    return {
        "model": worker_model,
        "tokenizer": worker_tokenizer,
        "selector": worker_selector,
        "fitness": worker_fitness,
    }


def _evaluate_chunk(bundle, chunk):
    out = []
    for idx, cand in chunk:
        bundle["selector"].set_flat_params(cand)
        fit = float(bundle["fitness"].evaluate(bundle["model"], bundle["tokenizer"]))
        out.append((idx, fit))
    return out


def evaluate_candidates(candidates, bundles, base_fitness):
    """Evaluiert Kandidaten sequenziell oder parallel ueber Worker-Modelle."""
    shared_dataset = _prepare_shared_subset(base_fitness)
    for b in bundles:
        bfit = b["fitness"]
        bfit._full_pool = getattr(base_fitness, "_full_pool", None)
        bfit._dataset = shared_dataset
        bfit._needs_reshuffle = False

    if len(bundles) == 1:
        return [fit for _, fit in _evaluate_chunk(bundles[0], list(enumerate(candidates)))]

    indexed = list(enumerate(candidates))
    chunks = [indexed[i::len(bundles)] for i in range(len(bundles))]
    results = []
    with ThreadPoolExecutor(max_workers=len(bundles)) as ex:
        futs = [ex.submit(_evaluate_chunk, bundles[i], chunks[i]) for i in range(len(bundles))]
        for f in futs:
            results.extend(f.result())

    results.sort(key=lambda x: x[0])
    return [fit for _, fit in results]


TRAIN_RUNS = []

run_name = TARGET_LAYER_CONFIG["name"]
run_dir = TRAIN_ROOT / run_name
model_dir = run_dir / "model"
run_dir.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 80)
print(f"Train run: {run_name}")
print(
    f"Layers: {TARGET_LAYER_CONFIG['layer_indices']} | "
    f"components={TARGET_LAYER_CONFIG['components']} ({TARGET_LAYER_CONFIG['expected_params']})"
)
print("=" * 80)

# Worker-Modelle bauen (Parallel-Eval)
worker_count = min(CANDIDATE_EVAL_WORKERS, ES_PARAMS["population_size"])
WORKER_BUNDLES = []
for wid in range(worker_count):
    WORKER_BUNDLES.append(_build_worker_bundle(wid + 1))

selector = WORKER_BUNDLES[0]["selector"]
model = WORKER_BUNDLES[0]["model"]
tokenizer = WORKER_BUNDLES[0]["tokenizer"]

es = OpenAIES(**ES_PARAMS)
base_fitness = build_train_fitness(reshuffle=True)

current_params = selector.get_flat_params().clone()
best_params = current_params.clone()
best_train = -float("inf")

history = []
t_run = time.time()

for gen in range(1, NUM_GENERATIONS + 1):
    candidates = es.ask(current_params)
    fitnesses = evaluate_candidates(candidates, WORKER_BUNDLES, base_fitness)

    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)

    if result.best_fitness > best_train:
        best_train = float(result.best_fitness)
        best_params = current_params.clone()

    row = {
        "gen": gen,
        "best_train": float(result.best_fitness),
        "mean_train": float(result.mean_fitness),
        "std_train": float(result.std_fitness),
        "best_train_ever": float(best_train),
        "sigma": float(es.current_sigma),
        "elapsed_min": (time.time() - t_run) / 60.0,
    }
    history.append(row)

    print(
        f"Gen {gen:02d}/{NUM_GENERATIONS} | "
        f"train_best={row['best_train']:.4f} "
        f"train_mean={row['mean_train']:.4f} "
        f"best_ever={row['best_train_ever']:.4f}"
    )

# Best state anwenden + speichern
selector.set_flat_params(best_params)
model_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(model_dir))
tokenizer.save_pretrained(str(model_dir))

elapsed = time.time() - t_run

summary = {
    "run_name": run_name,
    "train_task": TARGET_LAYER_CONFIG["train_task"],
    "layer_indices": TARGET_LAYER_CONFIG["layer_indices"],
    "components": TARGET_LAYER_CONFIG["components"],
    "expected_params": TARGET_LAYER_CONFIG["expected_params"],
    "selected_params": int(selector.num_target_elements),
    "num_generations_configured": NUM_GENERATIONS,
    "es_params": ES_PARAMS,
    "fitness_num_samples": FITNESS_NUM_SAMPLES,
    "fitness_batch_size": FITNESS_BATCH_SIZE,
    "candidate_eval_workers": worker_count,
    "history": history,
    "best_train": float(best_train),
    "elapsed_sec": float(elapsed),
    "model_dir": str(model_dir),
}
(run_dir / "train_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

TRAIN_RUNS.append(summary)

train_df = pd.DataFrame([
    {
        "run_name": summary["run_name"],
        "layers": str(summary["layer_indices"]),
        "components": summary["components"],
        "selected_params": summary["selected_params"],
        "best_train": summary["best_train"],
        "workers": summary["candidate_eval_workers"],
        "elapsed_min": summary["elapsed_sec"] / 60.0,
    }
])

display(train_df)

TRAIN_RUNS_PATH = TRAIN_ROOT / "all_train_runs.json"
TRAIN_RUNS_PATH.write_text(json.dumps(TRAIN_RUNS, indent=2), encoding="utf-8")

print(f"✅ Saved: {model_dir}")
print(f"⏱️ Elapsed: {elapsed/60:.1f} min")
print(f"Summary: {TRAIN_RUNS_PATH}")



In [ ]:
# 3.4 OLMES-Evaluation fuer alle trainingsmodelle auf GSM8K + HellaSwag

import os
import shutil
import matplotlib.pyplot as plt

script_path = Path("scripts/eval_olmes_tasks.py")
if not script_path.exists():
    raise FileNotFoundError("scripts/eval_olmes_tasks.py nicht gefunden.")


def _ensure_olmes_ready():
    """Installiert OLMES analog zur Baseline-Zelle, falls CLI fehlt."""
    if shutil.which("olmes") or shutil.which("oe-eval"):
        return

    olmes_dir = Path("/content/olmes")
    if not olmes_dir.exists():
        _run_cmd(["git", "clone", "https://github.com/allenai/olmes.git", str(olmes_dir)], check=True)
    else:
        _run_cmd(["git", "-C", str(olmes_dir), "pull"], check=True)

    _run_cmd(["python3", "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"], check=True)
    _run_cmd(["python3", "-m", "pip", "install", "-q", "-e", "."], check=True, cwd=str(olmes_dir))
    _run_cmd(["python3", "-m", "pip", "install", "-q", "-e", ".[gpu]"], check=False, cwd=str(olmes_dir))


if "TRAIN_RUNS" not in globals() or not TRAIN_RUNS:
    # Fallback fuer Kernel-Neustart
    train_runs_file = max(
        TRAIN_ROOT.glob("all_train_runs.json"),
        key=lambda p: p.stat().st_mtime,
    )
    TRAIN_RUNS = json.loads(train_runs_file.read_text(encoding="utf-8"))


def _run_cmd(cmd, check=True, cwd=None):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, text=True, capture_output=True, cwd=cwd)
    if p.stdout:
        print(p.stdout[-1200:])
    if p.returncode != 0 and p.stderr:
        print(p.stderr[-1200:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
    return p


def _read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _extract_task_entries(metrics_obj):
    if isinstance(metrics_obj.get("tasks"), list):
        return metrics_obj["tasks"]
    if isinstance(metrics_obj.get("tasks"), dict):
        out = []
        for name, val in metrics_obj["tasks"].items():
            if isinstance(val, dict):
                e = dict(val)
                e.setdefault("task_name", name)
                out.append(e)
        return out
    return []


def _pick_accuracy(task_name: str, row: dict):
    t = (task_name or "").lower()
    if "gsm8k" in t:
        pref = ["exact_match", "exact_match_flex", "primary_score", "acc_norm", "acc_raw"]
    elif "hellaswag" in t:
        pref = ["acc_norm", "acc_raw", "primary_score", "exact_match", "exact_match_flex"]
    else:
        pref = ["primary_score", "acc_norm", "acc_raw", "exact_match", "exact_match_flex"]

    for k in pref:
        v = row.get(k)
        if v is not None and pd.notna(v):
            return k, float(v)
    return None, None


def _load_single_task_accuracy(metrics_path: Path, expected_task: str):
    mobj = _read_json(metrics_path)
    entries = _extract_task_entries(mobj)
    if not entries:
        return None, None, None

    picked = entries[0]
    for e in entries:
        n = (e.get("alias") or e.get("task_name") or e.get("name") or "").lower()
        if expected_task.split("::")[0] in n:
            picked = e
            break

    metrics = picked.get("metrics", {}) or {}
    task_name = picked.get("alias") or picked.get("task_name") or picked.get("name") or expected_task
    row = {
        "primary_score": metrics.get("primary_score", picked.get("primary_score")),
        "acc_raw": metrics.get("acc_raw", picked.get("acc_raw")),
        "acc_norm": metrics.get("acc_norm", picked.get("acc_norm")),
        "exact_match": metrics.get("exact_match", picked.get("exact_match")),
        "exact_match_flex": metrics.get("exact_match_flex", picked.get("exact_match_flex")),
    }
    metric_name, acc = _pick_accuracy(task_name, row)
    return task_name, metric_name, acc


EVAL_TASKS = [
    ("gsm8k::olmes", "gsm8k"),
    ("hellaswag::olmo1", "hellaswag"),
]

_ensure_olmes_ready()

EVAL_ROWS = []

for run in TRAIN_RUNS:
    run_name = run["run_name"]
    model_dir = Path(run["model_dir"])
    eval_root = Path(model_dir).parent / "olmes_eval"
    eval_root.mkdir(parents=True, exist_ok=True)

    for eval_task, short_name in EVAL_TASKS:
        out_dir = eval_root / short_name
        out_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            "python3", str(script_path),
            "--model", str(model_dir),
            "--tasks", eval_task,
            "--model_type", "hf",
            "--limit", OLMES_LIMIT,
            "--num_shots", str(OLMES_NUM_SHOTS),
            "--batch_size", str(OLMES_BATCH_SIZE),
            "--cache_dir", OLMES_CACHE_DIR,
            "--output_dir", str(out_dir),
        ]
        _run_cmd(cmd, check=True)

        metrics_path = out_dir / "metrics.json"
        if not metrics_path.exists():
            EVAL_ROWS.append({
                "run_name": run_name,
                "train_task": run["train_task"],
                "layers": str(run["layer_indices"]),
                "components": run["components"],
                "eval_task": eval_task,
                "metric": None,
                "accuracy": None,
                "metrics_path": str(metrics_path),
            })
            continue

        task_name, metric_name, acc = _load_single_task_accuracy(metrics_path, eval_task)
        EVAL_ROWS.append({
            "run_name": run_name,
            "train_task": run["train_task"],
            "layers": str(run["layer_indices"]),
            "components": run["components"],
            "eval_task": task_name or eval_task,
            "metric": metric_name,
            "accuracy": acc,
            "metrics_path": str(metrics_path),
        })


eval_df = pd.DataFrame(EVAL_ROWS)
display(eval_df)

(TRAIN_ROOT / "all_olmes_eval_results.json").write_text(
    json.dumps(EVAL_ROWS, indent=2),
    encoding="utf-8",
)

# Kompakte Vergleichstabelle
pivot = eval_df.pivot_table(
    index=["run_name", "train_task", "layers", "components"],
    columns="eval_task",
    values="accuracy",
    aggfunc="first",
).reset_index()

display(pivot)

# Plot: Accuracy pro Run getrennt nach Eval-Task
plot_df = eval_df.dropna(subset=["accuracy"]).copy()
if not plot_df.empty:
    plt.figure(figsize=(12, 5))
    for eval_task, grp in plot_df.groupby("eval_task"):
        x = [f"{r}\n(train:{t})" for r, t in zip(grp["run_name"], grp["train_task"])]
        plt.plot(x, grp["accuracy"], marker="o", label=eval_task)
    plt.xticks(rotation=35, ha="right")
    plt.ylim(0, 1)
    plt.ylabel("accuracy")
    plt.title("OLMES Evaluation aller 3 Trainings-Setups auf GSM8K + HellaSwag")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

print(f"OLMES-Eval Summary gespeichert: {TRAIN_ROOT / 'all_olmes_eval_results.json'}")

### 3.5 Analog: ES-Training mit LogLikelihood-Fitness (gleiche Hyperparameter, gleiche 3 Layer-Setups)

Auch hier laufen 6 Trainingslaeufe:
- 3 Layer-Setups x 2 Trainingsaufgaben (`gsm8k`, `hellaswag`)

Unterschied zu 3.3:
- Fitness ist **LogLikelihood** statt 0/1-Accuracy.

In [ ]:
# 3.6 LL-Training: 3 Layer-Setups x 2 Aufgaben (GSM8K, HellaSwag)

from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness


class HellaSwagLogLikelihoodFitness:
    """Mittlere Token-LogLikelihood der korrekten HellaSwag-Fortsetzung."""

    def __init__(
        self,
        num_samples: int = 24,
        split: str = "train",
        seed: int = 42,
        reshuffle: bool = True,
        pool_size: int | None = 4000,
        batch_size: int = 1,
    ):
        self.num_samples = num_samples
        self.split = split
        self.seed = seed
        self.reshuffle = reshuffle
        self.pool_size = pool_size
        self.batch_size = batch_size

        self._rng = random.Random(seed)
        self._full_pool = None
        self._dataset = None
        self._needs_reshuffle = True

    def reshuffle_data(self):
        if self.reshuffle:
            self._needs_reshuffle = True

    def _load_data(self):
        if self._full_pool is None:
            ds = load_dataset("Rowan/hellaswag", split=self.split)
            ds = ds.shuffle(seed=self.seed)
            if self.pool_size is not None:
                ds = ds.select(range(min(self.pool_size, len(ds))))
            self._full_pool = ds

        if self.reshuffle and self._needs_reshuffle:
            indices = self._rng.sample(
                range(len(self._full_pool)),
                min(self.num_samples, len(self._full_pool)),
            )
            self._dataset = self._full_pool.select(indices)
            self._needs_reshuffle = False
        elif self._dataset is None:
            self._dataset = self._full_pool.select(
                range(min(self.num_samples, len(self._full_pool)))
            )

    @staticmethod
    def _mean_ll_for_suffix(model, tokenizer, prompt: str, suffix: str) -> float:
        full = prompt + suffix
        tok_full = tokenizer(full, return_tensors="pt").to(model.device)
        tok_prompt = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model(**tok_full)
            logits = out.logits[:, :-1, :]
            labels = tok_full["input_ids"][:, 1:]
            log_probs = torch.log_softmax(logits, dim=-1)
            token_log_probs = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

        prompt_len = tok_prompt["input_ids"].shape[1] - 1
        suffix_ll = token_log_probs[:, prompt_len:]
        if suffix_ll.numel() == 0:
            return -1e9
        return float(suffix_ll.mean().item())

    @torch.inference_mode()
    def evaluate(self, model, tokenizer) -> float:
        self._load_data()
        model.eval()

        total_ll = 0.0
        count = 0
        for ex in tqdm(self._dataset, desc="    HellaSwag LL", unit="q", leave=False):
            endings = ex.get("endings") or []
            if not endings:
                continue

            gold_idx = int(ex["label"])
            if gold_idx < 0 or gold_idx >= len(endings):
                continue

            prompt = (
                "Choose the most plausible continuation for the context.\n"
                f"Context: {ex.get('ctx', '')}\n"
                "Continuation: "
            )
            gold_suffix = endings[gold_idx]
            ll = self._mean_ll_for_suffix(model, tokenizer, prompt, gold_suffix)
            total_ll += ll
            count += 1

        return total_ll / count if count > 0 else -float("inf")


def build_ll_fitness(task_name: str):
    if task_name == "gsm8k":
        return GSM8KLogLikelihoodFitness(
            num_samples=FITNESS_NUM_SAMPLES,
            split="train",
            target_mode="short",
            seed=SEED,
            batch_size=4,
            reshuffle=True,
            pool_size=4000,
        )
    if task_name == "hellaswag":
        return HellaSwagLogLikelihoodFitness(
            num_samples=FITNESS_NUM_SAMPLES,
            split="train",
            seed=SEED,
            reshuffle=True,
            pool_size=4000,
            batch_size=1,
        )
    raise ValueError(f"Unbekannter Task: {task_name}")


LL_TRAIN_ROOT = OUTPUT_BASE / f"es_ll_training_{RUN_TAG}"
LL_TRAIN_ROOT.mkdir(parents=True, exist_ok=True)
LL_TRAIN_RUNS = []

for train_task in TRAIN_TASKS:
    for layer_cfg in LAYER_CONFIGS:
        run_name = f"{train_task}__{layer_cfg.name}"
        run_dir = LL_TRAIN_ROOT / run_name
        model_dir = run_dir / "model"
        run_dir.mkdir(parents=True, exist_ok=True)

        print("\n" + "=" * 80)
        print(f"LL train run: {run_name}")
        print(f"Layers: {layer_cfg.layer_indices} | components={layer_cfg.components} ({layer_cfg.expected_params})")
        print("=" * 80)

        model, tokenizer = load_model_and_tokenizer(
            model_name=MODEL_NAME,
            dtype=DTYPE,
            device=DEVICE,
        )

        selector = LayerSelector(
            model=model,
            strategy="by_index",
            layer_indices=layer_cfg.layer_indices,
            components=layer_cfg.components,
        )
        print(f"Selected params: {selector.num_target_elements:,}")

        es = OpenAIES(**ES_PARAMS)
        fitness = build_ll_fitness(train_task)

        current_params = selector.get_flat_params().clone()
        best_params = current_params.clone()
        best_ever = -float("inf")
        history = []

        t_run = time.time()
        for gen in range(1, NUM_GENERATIONS + 1):
            if hasattr(fitness, "reshuffle_data"):
                fitness.reshuffle_data()

            candidates = es.ask(current_params)
            fitnesses = []
            for cand in candidates:
                selector.set_flat_params(cand)
                fit = float(fitness.evaluate(model, tokenizer))
                fitnesses.append(fit)

            result = es.tell(current_params, candidates, fitnesses)
            current_params = result.new_params.clone()
            selector.set_flat_params(current_params)

            if result.best_fitness > best_ever:
                best_ever = result.best_fitness
                best_params = current_params.clone()

            row = {
                "gen": gen,
                "best": float(result.best_fitness),
                "mean": float(result.mean_fitness),
                "std": float(result.std_fitness),
                "best_ever": float(best_ever),
                "sigma": float(es.current_sigma),
            }
            history.append(row)
            print(
                f"Gen {gen:02d}/{NUM_GENERATIONS} | best={row['best']:.4f} "
                f"mean={row['mean']:.4f} best_ever={row['best_ever']:.4f}"
            )

        selector.set_flat_params(best_params)
        model_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(model_dir))
        tokenizer.save_pretrained(str(model_dir))

        elapsed = time.time() - t_run
        summary = {
            "run_name": run_name,
            "train_task": train_task,
            "layer_indices": layer_cfg.layer_indices,
            "components": layer_cfg.components,
            "expected_params": layer_cfg.expected_params,
            "selected_params": int(selector.num_target_elements),
            "num_generations": NUM_GENERATIONS,
            "es_params": ES_PARAMS,
            "fitness_type": "loglikelihood",
            "fitness_num_samples": FITNESS_NUM_SAMPLES,
            "history": history,
            "best_ever": float(best_ever),
            "elapsed_sec": float(elapsed),
            "model_dir": str(model_dir),
        }
        (run_dir / "train_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

        LL_TRAIN_RUNS.append(summary)

        del model, tokenizer
        torch.cuda.empty_cache()

        print(f"✅ Saved: {model_dir}")
        print(f"⏱️  Elapsed: {elapsed/60:.1f} min | best fitness={best_ever:.4f}")

LL_TRAIN_RUNS_PATH = LL_TRAIN_ROOT / "all_train_runs.json"
LL_TRAIN_RUNS_PATH.write_text(json.dumps(LL_TRAIN_RUNS, indent=2), encoding="utf-8")

ll_train_df = pd.DataFrame([
    {
        "run_name": r["run_name"],
        "train_task": r["train_task"],
        "layers": str(r["layer_indices"]),
        "components": r["components"],
        "selected_params": r["selected_params"],
        "best_ever_ll": r["best_ever"],
        "elapsed_min": r["elapsed_sec"] / 60.0,
    }
    for r in LL_TRAIN_RUNS
])

display(ll_train_df.sort_values(["train_task", "selected_params"]))
print(f"\nAlle LL-Trainings fertig. Summary: {LL_TRAIN_RUNS_PATH}")

In [ ]:
# 3.7 OLMES-Evaluation fuer LL-trainierte Modelle

import shutil

if "LL_TRAIN_RUNS" not in globals() or not LL_TRAIN_RUNS:
    ll_runs_file = max(
        LL_TRAIN_ROOT.glob("all_train_runs.json"),
        key=lambda p: p.stat().st_mtime,
    )
    LL_TRAIN_RUNS = json.loads(ll_runs_file.read_text(encoding="utf-8"))

# Falls Zelle 14 nicht zuvor lief: OLMES-Helfer lokal bereitstellen.
if "_run_cmd" not in globals():
    def _run_cmd(cmd, check=True, cwd=None):
        print("$", " ".join(cmd))
        p = subprocess.run(cmd, text=True, capture_output=True, cwd=cwd)
        if p.stdout:
            print(p.stdout[-1200:])
        if p.returncode != 0 and p.stderr:
            print(p.stderr[-1200:])
        if check and p.returncode != 0:
            raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
        return p

if "_ensure_olmes_ready" not in globals():
    def _ensure_olmes_ready():
        if shutil.which("olmes") or shutil.which("oe-eval"):
            return
        olmes_dir = Path("/content/olmes")
        if not olmes_dir.exists():
            _run_cmd(["git", "clone", "https://github.com/allenai/olmes.git", str(olmes_dir)], check=True)
        else:
            _run_cmd(["git", "-C", str(olmes_dir), "pull"], check=True)
        _run_cmd(["python3", "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"], check=True)
        _run_cmd(["python3", "-m", "pip", "install", "-q", "-e", "."], check=True, cwd=str(olmes_dir))
        _run_cmd(["python3", "-m", "pip", "install", "-q", "-e", ".[gpu]"], check=False, cwd=str(olmes_dir))

if "_load_single_task_accuracy" not in globals():
    def _read_json(path: Path):
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)

    def _extract_task_entries(metrics_obj):
        if isinstance(metrics_obj.get("tasks"), list):
            return metrics_obj["tasks"]
        if isinstance(metrics_obj.get("tasks"), dict):
            out = []
            for name, val in metrics_obj["tasks"].items():
                if isinstance(val, dict):
                    e = dict(val)
                    e.setdefault("task_name", name)
                    out.append(e)
            return out
        return []

    def _pick_accuracy(task_name: str, row: dict):
        t = (task_name or "").lower()
        if "gsm8k" in t:
            pref = ["exact_match", "exact_match_flex", "primary_score", "acc_norm", "acc_raw"]
        elif "hellaswag" in t:
            pref = ["acc_norm", "acc_raw", "primary_score", "exact_match", "exact_match_flex"]
        else:
            pref = ["primary_score", "acc_norm", "acc_raw", "exact_match", "exact_match_flex"]
        for k in pref:
            v = row.get(k)
            if v is not None and pd.notna(v):
                return k, float(v)
        return None, None

    def _load_single_task_accuracy(metrics_path: Path, expected_task: str):
        mobj = _read_json(metrics_path)
        entries = _extract_task_entries(mobj)
        if not entries:
            return None, None, None
        picked = entries[0]
        for e in entries:
            n = (e.get("alias") or e.get("task_name") or e.get("name") or "").lower()
            if expected_task.split("::")[0] in n:
                picked = e
                break
        metrics = picked.get("metrics", {}) or {}
        task_name = picked.get("alias") or picked.get("task_name") or picked.get("name") or expected_task
        row = {
            "primary_score": metrics.get("primary_score", picked.get("primary_score")),
            "acc_raw": metrics.get("acc_raw", picked.get("acc_raw")),
            "acc_norm": metrics.get("acc_norm", picked.get("acc_norm")),
            "exact_match": metrics.get("exact_match", picked.get("exact_match")),
            "exact_match_flex": metrics.get("exact_match_flex", picked.get("exact_match_flex")),
        }
        metric_name, acc = _pick_accuracy(task_name, row)
        return task_name, metric_name, acc

if "EVAL_TASKS" not in globals():
    EVAL_TASKS = [
        ("gsm8k::olmes", "gsm8k"),
        ("hellaswag::olmo1", "hellaswag"),
    ]

_ensure_olmes_ready()

LL_EVAL_ROWS = []
for run in LL_TRAIN_RUNS:
    run_name = run["run_name"]
    model_dir = Path(run["model_dir"])
    eval_root = Path(model_dir).parent / "olmes_eval"
    eval_root.mkdir(parents=True, exist_ok=True)

    for eval_task, short_name in EVAL_TASKS:
        out_dir = eval_root / short_name
        out_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            "python3", str(script_path),
            "--model", str(model_dir),
            "--tasks", eval_task,
            "--model_type", "hf",
            "--limit", OLMES_LIMIT,
            "--num_shots", str(OLMES_NUM_SHOTS),
            "--batch_size", str(OLMES_BATCH_SIZE),
            "--cache_dir", OLMES_CACHE_DIR,
            "--output_dir", str(out_dir),
        ]
        _run_cmd(cmd, check=True)

        metrics_path = out_dir / "metrics.json"
        if not metrics_path.exists():
            LL_EVAL_ROWS.append({
                "run_name": run_name,
                "train_task": run["train_task"],
                "layers": str(run["layer_indices"]),
                "components": run["components"],
                "eval_task": eval_task,
                "metric": None,
                "accuracy": None,
                "metrics_path": str(metrics_path),
            })
            continue

        task_name, metric_name, acc = _load_single_task_accuracy(metrics_path, eval_task)
        LL_EVAL_ROWS.append({
            "run_name": run_name,
            "train_task": run["train_task"],
            "layers": str(run["layer_indices"]),
            "components": run["components"],
            "eval_task": task_name or eval_task,
            "metric": metric_name,
            "accuracy": acc,
            "metrics_path": str(metrics_path),
        })

ll_eval_df = pd.DataFrame(LL_EVAL_ROWS)
display(ll_eval_df)

(LL_TRAIN_ROOT / "all_olmes_eval_results.json").write_text(
    json.dumps(LL_EVAL_ROWS, indent=2),
    encoding="utf-8",
)

ll_pivot = ll_eval_df.pivot_table(
    index=["run_name", "train_task", "layers", "components"],
    columns="eval_task",
    values="accuracy",
    aggfunc="first",
).reset_index()
display(ll_pivot)

print(f"LL OLMES-Eval Summary gespeichert: {LL_TRAIN_ROOT / 'all_olmes_eval_results.json'}")

In [ ]:
# 3.8 Gesamtvergleich: Base vs Binary vs LogLikelihood (alle trainierten Modelle)


def _canon_task(task_name: str) -> str:
    t = (task_name or "").lower()
    if "gsm8k" in t:
        return "gsm8k"
    if "hellaswag" in t:
        return "hellaswag"
    return t


def _find_latest_baseline_root() -> Path | None:
    roots = [
        Path("/content/experiments/olmes"),
        Path(globals().get("OUTPUT_BASE", "/content/experiments")) / "olmes",
        Path(globals().get("OUTPUT_BASE", "/content/experiments")),
    ]

    candidates = []
    for root in roots:
        if not root.exists():
            continue
        candidates.extend(root.glob("olmes_baseline_gsm8k_hellaswag_*"))

    candidates = sorted(
        [c for c in candidates if c.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None


def _baseline_task_metrics(root: Path) -> dict:
    mapping = {}
    summary_path = root / "combined_summary.json"

    task_paths = {
        "gsm8k": root / "gsm8k" / "metrics.json",
        "hellaswag": root / "hellaswag" / "metrics.json",
    }

    if summary_path.exists():
        summary = _read_json(summary_path)
        to = summary.get("task_outputs", {})
        for tk, p in to.items():
            if "gsm8k" in tk:
                task_paths["gsm8k"] = Path(p) / "metrics.json"
            if "hellaswag" in tk:
                task_paths["hellaswag"] = Path(p) / "metrics.json"

    for short_task, mpath in task_paths.items():
        if not mpath.exists():
            mapping[short_task] = None
            continue
        tname, mname, acc = _load_single_task_accuracy(
            mpath,
            "gsm8k::olmes" if short_task == "gsm8k" else "hellaswag::olmo1",
        )
        mapping[short_task] = {
            "task_name": tname,
            "metric": mname,
            "accuracy": acc,
            "metrics_path": str(mpath),
        }

    return mapping


# Eingaben
binary_eval_df = eval_df.copy()
loglik_eval_df = ll_eval_df.copy()

for df in [binary_eval_df, loglik_eval_df]:
    df["eval_task_short"] = df["eval_task"].map(_canon_task)
    df["run_key"] = (
        df["train_task"].astype(str)
        + "|"
        + df["layers"].astype(str)
        + "|"
        + df["components"].astype(str)
    )

baseline_root = _find_latest_baseline_root()
if baseline_root is None:
    print("Hinweis: Keine Baseline gefunden. Base-Spalten werden als n/a gefuellt.")
    baseline_map = {"gsm8k": {"accuracy": None}, "hellaswag": {"accuracy": None}}
else:
    baseline_map = _baseline_task_metrics(baseline_root)

# Binary/LL je Run+Task zusammenfuehren
bin_small = binary_eval_df[["run_key", "run_name", "train_task", "layers", "components", "eval_task_short", "accuracy"]].rename(columns={"accuracy": "binary_acc"})
ll_small = loglik_eval_df[["run_key", "run_name", "train_task", "layers", "components", "eval_task_short", "accuracy"]].rename(columns={"accuracy": "loglik_acc"})

cmp_df = pd.merge(
    bin_small,
    ll_small[["run_key", "eval_task_short", "loglik_acc"]],
    on=["run_key", "eval_task_short"],
    how="outer",
)

# Base-Accuracy pro Task anhängen
cmp_df["base_acc"] = cmp_df["eval_task_short"].map(
    lambda t: (baseline_map.get(t) or {}).get("accuracy")
)

cmp_df["delta_binary_vs_base"] = cmp_df["binary_acc"] - cmp_df["base_acc"]
cmp_df["delta_loglik_vs_base"] = cmp_df["loglik_acc"] - cmp_df["base_acc"]
cmp_df["delta_loglik_vs_binary"] = cmp_df["loglik_acc"] - cmp_df["binary_acc"]

cmp_df = cmp_df.sort_values(["eval_task_short", "train_task", "layers"]).reset_index(drop=True)
display(cmp_df)

# Kompakt-Pivot fuer schnelleren Vergleich
pivot_cmp = cmp_df.pivot_table(
    index=["train_task", "layers", "components", "run_name"],
    columns="eval_task_short",
    values=["base_acc", "binary_acc", "loglik_acc", "delta_binary_vs_base", "delta_loglik_vs_base", "delta_loglik_vs_binary"],
    aggfunc="first",
)
display(pivot_cmp)

# Plot (je Task): Base-Linie + Binary/LL pro Run
plot_cmp = cmp_df.dropna(subset=["binary_acc", "loglik_acc"]).copy()
if not plot_cmp.empty:
    for task_name, grp in plot_cmp.groupby("eval_task_short"):
        plt.figure(figsize=(12, 4))
        x = [f"{r}\n(train:{t})" for r, t in zip(grp["run_name"], grp["train_task"])]
        plt.plot(x, grp["binary_acc"], marker="o", label="binary")
        plt.plot(x, grp["loglik_acc"], marker="o", label="loglikelihood")

        base_val = grp["base_acc"].dropna()
        if not base_val.empty:
            plt.axhline(float(base_val.iloc[0]), linestyle="--", color="red", alpha=0.6, label="base")

        plt.ylim(0, 1)
        plt.xticks(rotation=35, ha="right")
        plt.ylabel("accuracy")
        plt.title(f"Vergleich auf {task_name}: base vs binary vs loglikelihood")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

cmp_out = TRAIN_ROOT / "comparison_base_binary_loglik.json"
cmp_out.write_text(cmp_df.to_json(orient="records", indent=2), encoding="utf-8")
print("Baseline root:", baseline_root if baseline_root is not None else "n/a")
print("Vergleich gespeichert:", cmp_out)

## Kombinierte Variante

In [ ]:
# Kombiniertes Training: Softprompt (Kollege) + Option D v5 Layer-wise ES
print("Hinweis: Hier wird der Softprompt als Embedding-Prefix genutzt (kein Decoding zu Text).")

import json
import random
import time
from pathlib import Path

import torch
import pandas as pd

from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness


# -----------------------------------------------------------------------------
# 1) Setup
# -----------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = globals().get("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16" if DEVICE.startswith("cuda") else "float32"

# Option D v5 Hyperparameter (wie von dir vorgegeben)
NUM_GENERATIONS = 100
LAYER_INDICES = [18, 19, 20, 21]

# Run-Ausgabe
global_output_base = Path(globals().get("OUTPUT_BASE", "/content/experiments"))
run_tag = time.strftime("%Y%m%d_%H%M%S")
hybrid_run_dir = global_output_base / f"hybrid_softprompt_option_d_v5_{run_tag}"
hybrid_run_dir.mkdir(parents=True, exist_ok=True)


def _resolve_prompt_file() -> Path:
    explicit = globals().get("SOFTPROMPT_FILE", None)
    candidates = []
    if explicit:
        candidates.append(Path(explicit))

    if "PROJECT_DIR" in globals():
        candidates.append(Path(globals()["PROJECT_DIR"]) / "best_prompt_overall.pt")

    candidates.extend(
        [
            Path.cwd() / "best_prompt_overall.pt",
            Path("/content/PG_ML_AI/es-llm-finetune-Hasan-Dev/best_prompt_overall.pt"),
            Path("/home/hasan/DEV/Uni/Hub_PG_MLAI/PG_ML_AI/es-llm-finetune-Hasan-Dev/best_prompt_overall.pt"),
        ]
    )

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        "best_prompt_overall.pt nicht gefunden. Setze SOFTPROMPT_FILE auf den korrekten Pfad."
    )


def _load_soft_prompt_tensor(path: Path) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, torch.Tensor):
        t = obj
    elif isinstance(obj, dict):
        t = None
        for k in ["soft_prompt", "prompt", "best_prompt", "tensor", "embeddings"]:
            if k in obj and isinstance(obj[k], torch.Tensor):
                t = obj[k]
                break
        if t is None:
            raise ValueError(f"Keine Tensor-Daten im Dict gefunden. Keys: {list(obj.keys())}")
    else:
        raise ValueError(f"Unbekanntes Prompt-Format: {type(obj)}")

    if t.ndim == 3 and t.shape[0] == 1:
        t = t.squeeze(0)
    if t.ndim != 2:
        raise ValueError(f"Softprompt muss 2D sein [prompt_len, hidden_dim], erhalten: {tuple(t.shape)}")
    return t.float().cpu()


class SoftPromptGSM8KLogLikelihoodFitness(GSM8KLogLikelihoodFitness):
    """LL-Fitness mit Softprompt-Embedding als Prefix (kein Text-Decoding)."""

    def __init__(self, soft_prompt: torch.Tensor, **kwargs):
        super().__init__(**kwargs)
        if soft_prompt.ndim != 2:
            raise ValueError(f"soft_prompt muss 2D sein, erhalten: {tuple(soft_prompt.shape)}")
        self.soft_prompt = soft_prompt.float().cpu()

    @torch.inference_mode()
    def evaluate(self, model, tokenizer) -> float:
        self._load_data()
        model.eval()

        if self._cached_inputs is None:
            self._prepare_inputs(tokenizer)

        if not self._cached_inputs:
            return -float("inf")

        device = next(model.parameters()).device
        input_embed_layer = model.get_input_embeddings()
        embed_dtype = input_embed_layer.weight.dtype
        hidden_dim = input_embed_layer.weight.shape[1]

        if self.soft_prompt.shape[1] != hidden_dim:
            raise ValueError(
                f"Softprompt hidden_dim={self.soft_prompt.shape[1]} passt nicht zum Modell hidden_dim={hidden_dim}"
            )

        soft_prefix = self.soft_prompt.to(device=device, dtype=embed_dtype)
        prefix_len = soft_prefix.shape[0]

        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
        total_ll = 0.0
        count = 0

        for batch_start in range(0, len(self._cached_inputs), self.batch_size):
            batch = self._cached_inputs[batch_start : batch_start + self.batch_size]
            max_len = max(item["full_ids"].shape[0] for item in batch)

            input_ids = torch.full(
                (len(batch), max_len), pad_id, dtype=torch.long, device=device
            )
            attention_mask = torch.zeros(
                (len(batch), max_len), dtype=torch.long, device=device
            )

            for i, item in enumerate(batch):
                ids = item["full_ids"]
                input_ids[i, : ids.shape[0]] = ids.to(device)
                attention_mask[i, : ids.shape[0]] = 1

            token_embeds = input_embed_layer(input_ids)
            prefix_batch = soft_prefix.unsqueeze(0).expand(len(batch), -1, -1)
            inputs_embeds = torch.cat([prefix_batch, token_embeds], dim=1)

            prefix_attn = torch.ones((len(batch), prefix_len), dtype=torch.long, device=device)
            attention_mask_ext = torch.cat([prefix_attn, attention_mask], dim=1)

            logits = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask_ext).logits
            log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

            for i, item in enumerate(batch):
                ans_start = item["answer_start"]
                seq_len = item["full_ids"].shape[0]

                # Durch Prefix verschiebt sich die Logit-Position um prefix_len.
                ans_log_probs = log_probs[i, prefix_len + ans_start - 1 : prefix_len + seq_len - 1, :]
                ans_tokens = input_ids[i, ans_start:seq_len]
                token_lls = ans_log_probs.gather(1, ans_tokens.unsqueeze(1)).squeeze(1)
                total_ll += token_lls.mean().item()
                count += 1

        return total_ll / count if count > 0 else -float("inf")


# -----------------------------------------------------------------------------
# 2) Modell laden + Softprompt als Embedding vorbereiten
# -----------------------------------------------------------------------------
print("Lade Modell + Tokenizer ...")
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    dtype=DTYPE,
    device=DEVICE,
)

prompt_file = _resolve_prompt_file()
print("Nutze Softprompt-Datei:", prompt_file)

soft_prompt_tensor = _load_soft_prompt_tensor(prompt_file)
model_hidden = model.get_input_embeddings().weight.shape[1]
if soft_prompt_tensor.shape[1] != model_hidden:
    raise ValueError(
        f"Dimension mismatch: softprompt hidden={soft_prompt_tensor.shape[1]} vs model hidden={model_hidden}"
    )

softprompt_info = {
    "file": str(prompt_file),
    "shape": list(soft_prompt_tensor.shape),
    "dtype": str(soft_prompt_tensor.dtype),
    "mean_abs": float(soft_prompt_tensor.abs().mean().item()),
    "l2_mean_per_token": float(torch.norm(soft_prompt_tensor, dim=1).mean().item()),
}

(softprompt_info_path := hybrid_run_dir / "softprompt_embedding_info.json").write_text(
    json.dumps(softprompt_info, indent=2, ensure_ascii=False), encoding="utf-8"
 )

print("Softprompt als Embedding geladen.")
print("Shape:", tuple(soft_prompt_tensor.shape))
print("Info gespeichert:", softprompt_info_path)


# -----------------------------------------------------------------------------
# 3) Option D v5: Layer-wise ES Training (EXAKT dein Setup)
# -----------------------------------------------------------------------------
# 3.1) Layer auswaehlen - Attention von Layer 18-21 (~7.2M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=LAYER_INDICES,
    components="attention",
)
print(f"Target: {selector.num_target_elements:,} Parameter")
print(selector.summary())

# 3.2) ES mit Adam-Optimizer + Cosine Sigma-Schedule
es = OpenAIES(
    population_size=96,
    sigma=0.003,
    learning_rate=0.0008,
    antithetic=True,
    fitness_shaping="centered_rank",
    weight_decay=0.002,
    optimizer="adam",
    sigma_decay="cosine",
    sigma_final=0.001,
    max_generations=NUM_GENERATIONS,
)

# 3.3) Fitness: Full-Reasoning LL mit Embedding-Softprompt + per-Generation Reshuffling
fitness_fn = SoftPromptGSM8KLogLikelihoodFitness(
    soft_prompt=soft_prompt_tensor,
    num_samples=120,
    split="train",
    target_mode="full",
    batch_size=12,
    reshuffle=True,
    pool_size=5000,
)

# 3.4) Training Loop
current_params = selector.get_flat_params().clone()
best_ever = -float("inf")
best_ever_params = current_params.clone()
history = []

print("\n" + "=" * 70)
print(f"  ES-Training v5 (Hybrid Softprompt-Embedding): {NUM_GENERATIONS} Generationen, Pop={es.population_size}")
print(f"  Optimizer: Adam (beta1={es.adam_beta1}, beta2={es.adam_beta2})")
print(f"  Sigma-Schedule: cosine {es.sigma:.4f} -> {es.sigma_final:.4f}")
print(f"  Target: {selector.num_target_elements:,} Parameter (Layer 18-21 Attention)")
print(f"  Fitness: Full-Reasoning LL + Embedding-Prefix (n={fitness_fn.num_samples}, batch={fitness_fn.batch_size})")
print(f"  Softprompt Prefix: len={soft_prompt_tensor.shape[0]}, hidden={soft_prompt_tensor.shape[1]}")
print("  Data: Reshuffle per GENERATION - 120 gleiche Fragen fuer alle Kandidaten")
if torch.cuda.is_available():
    print(f"  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB belegt")
print("=" * 70 + "\n")

t_total = time.time()

for gen in range(1, NUM_GENERATIONS + 1):
    t0 = time.time()

    # Neue Fragen fuer diese Generation
    fitness_fn.reshuffle_data()

    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)

    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)

    if result.best_fitness > best_ever:
        best_ever = result.best_fitness
        best_ever_params = current_params.clone()

    elapsed = time.time() - t0
    history.append(
        {
            "gen": gen,
            "best": result.best_fitness,
            "mean": result.mean_fitness,
            "std": result.std_fitness,
            "best_ever": best_ever,
            "sigma": es.current_sigma,
            "elapsed": elapsed,
        }
    )

    if gen % 10 == 0 or gen == 1:
        total_t = time.time() - t_total
        eta = (total_t / gen) * (NUM_GENERATIONS - gen)
        print(
            f"Gen {gen:3d}/{NUM_GENERATIONS} | best={result.best_fitness:.4f}  "
            f"mean={result.mean_fitness:.4f}  best_ever={best_ever:.4f}  "
            f"sigma={es.current_sigma:.5f}  [{elapsed:.1f}s/gen, {total_t:.0f}s total, "
            f"ETA ~{eta/60:.0f}min]"
        )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Best-ever Parameter anwenden
selector.set_flat_params(best_ever_params)
total_time = time.time() - t_total
print(f"\nTraining abgeschlossen in {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Best-ever LL Fitness: {best_ever:.4f}")


# -----------------------------------------------------------------------------
# 4) Speichern
# -----------------------------------------------------------------------------
hybrid_model_dir = hybrid_run_dir / "best_model"
hybrid_model_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(hybrid_model_dir)
tokenizer.save_pretrained(hybrid_model_dir)

history_df = pd.DataFrame(history)
history_path = hybrid_run_dir / "hybrid_option_d_v5_history.csv"
history_df.to_csv(history_path, index=False)

cfg_payload = {
    "mode": "hybrid_softprompt_option_d_v5_embedding_prefix",
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "dtype": DTYPE,
    "layer_indices": LAYER_INDICES,
    "components": "attention",
    "num_generations": NUM_GENERATIONS,
    "population_size": 96,
    "sigma": 0.003,
    "learning_rate": 0.0008,
    "weight_decay": 0.002,
    "optimizer": "adam",
    "sigma_decay": "cosine",
    "sigma_final": 0.001,
    "fitness": {
        "name": "SoftPromptGSM8KLogLikelihoodFitness",
        "num_samples": 120,
        "split": "train",
        "target_mode": "full",
        "batch_size": 12,
        "reshuffle": True,
        "pool_size": 5000,
    },
    "softprompt": softprompt_info,
    "best_ever": float(best_ever),
    "run_dir": str(hybrid_run_dir),
}

(config_path := hybrid_run_dir / "hybrid_option_d_v5_config.json").write_text(
    json.dumps(cfg_payload, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("\nArtefakte:")
print("- run dir:", hybrid_run_dir)
print("- model:", hybrid_model_dir)
print("- history:", history_path)
print("- config:", config_path)
print("- softprompt info:", softprompt_info_path)

# Fuer Folgezellen
globals()["hybrid_run_dir"] = str(hybrid_run_dir)
globals()["hybrid_model_dir"] = str(hybrid_model_dir)
globals()["hybrid_history_df"] = history_df
globals()["hybrid_prompt_file"] = str(prompt_file)
globals()["hybrid_softprompt_tensor_shape"] = tuple(soft_prompt_tensor.shape)

### Hybrid Option D v5: GSM8K Evaluation + Visualisierung

Dieser Block evaluiert das trainierte Hybrid-Modell gegen das Base-Modell auf GSM8K (Test) und visualisiert:
- Training-Verlauf (LL-Fitness)
- Accuracy-Vergleich (Base vs Hybrid)
- LogLikelihood-Vergleich (Base vs Hybrid)

In [ ]:
# GSM8K Eval: Base vs Hybrid Option D v5 + Visualisierung

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from es_llm.model.loader import load_model_and_tokenizer
from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness


if "hybrid_model_dir" not in globals():
    raise RuntimeError("Bitte zuerst die Hybrid-Trainingszelle ausfuehren.")

MODEL_NAME = globals().get("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16" if DEVICE.startswith("cuda") else "float32"

EVAL_N = 200

print("Lade Base-Modell ...")
base_model, base_tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    dtype=DTYPE,
    device=DEVICE,
)
base_model.eval()

print("Lade Hybrid Fine-Tuned Modell ...")
ft_model, ft_tokenizer = load_model_and_tokenizer(
    model_name=str(hybrid_model_dir),
    dtype=DTYPE,
    device=DEVICE,
)
ft_model.eval()

# 1) Accuracy-Eval auf GSM8K Test
eval_acc = GSM8KFitness(
    num_samples=EVAL_N,
    split="test",
    max_new_tokens=256,
    seed=42,
    reshuffle=False,
)

print("\nEval Accuracy (Base) ...")
base_acc = float(eval_acc.evaluate(base_model, base_tokenizer))
print(f"Base Accuracy: {base_acc:.4f}")

print("Eval Accuracy (Hybrid FT) ...")
ft_acc = float(eval_acc.evaluate(ft_model, ft_tokenizer))
print(f"Hybrid Accuracy: {ft_acc:.4f}")

# 2) LogLikelihood-Eval auf GSM8K Test
eval_ll = GSM8KLogLikelihoodFitness(
    num_samples=EVAL_N,
    split="test",
    target_mode="full",
    seed=42,
    batch_size=8,
    reshuffle=False,
    pool_size=None,
)

print("\nEval LogLikelihood (Base) ...")
base_ll = float(eval_ll.evaluate(base_model, base_tokenizer))
print(f"Base LL: {base_ll:.4f}")

print("Eval LogLikelihood (Hybrid FT) ...")
ft_ll = float(eval_ll.evaluate(ft_model, ft_tokenizer))
print(f"Hybrid LL: {ft_ll:.4f}")

acc_delta = ft_acc - base_acc
ll_delta = ft_ll - base_ll

print("\nSummary")
print(f"- Accuracy Delta (FT - Base): {acc_delta:+.4f}")
print(f"- LogLikelihood Delta (FT - Base): {ll_delta:+.4f}")

# Ergebnisse speichern
run_dir = Path(globals().get("hybrid_run_dir", Path(hybrid_model_dir).parent))
eval_payload = {
    "eval_samples": EVAL_N,
    "base": {"accuracy": base_acc, "loglikelihood": base_ll},
    "hybrid_finetuned": {"accuracy": ft_acc, "loglikelihood": ft_ll},
    "delta": {"accuracy": acc_delta, "loglikelihood": ll_delta},
}

eval_path = run_dir / "hybrid_option_d_v5_gsm8k_eval.json"
with eval_path.open("w", encoding="utf-8") as f:
    json.dump(eval_payload, f, indent=2)

print("Eval gespeichert:", eval_path)

# Training-History laden (falls nicht im RAM)
hist_df = globals().get("hybrid_history_df", None)
if hist_df is None or len(hist_df) == 0:
    hist_csv = run_dir / "hybrid_option_d_v5_history.csv"
    if hist_csv.exists():
        hist_df = pd.read_csv(hist_csv)

# Visualisierung
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if hist_df is not None and len(hist_df) > 0:
    axes[0].plot(hist_df["gen"], hist_df["best"], label="best", linewidth=2)
    axes[0].plot(hist_df["gen"], hist_df["mean"], label="mean", linewidth=2)
    axes[0].plot(hist_df["gen"], hist_df["best_ever"], label="best_ever", linewidth=2, linestyle="--")
    axes[0].set_title("Hybrid Option D v5: Training-Verlauf")
    axes[0].set_xlabel("Generation")
    axes[0].set_ylabel("LL Fitness")
    axes[0].grid(alpha=0.3)
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "Keine History gefunden", ha="center", va="center")
    axes[0].set_axis_off()

acc_vals = [base_acc, ft_acc]
axes[1].bar(["Base", "Hybrid FT"], acc_vals, color=["#7a7a7a", "#2a9d8f"])
axes[1].set_title(f"GSM8K Accuracy (n={EVAL_N})")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, max(0.1, min(1.0, max(acc_vals) + 0.08)))
for i, v in enumerate(acc_vals):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha="center")

ll_vals = [base_ll, ft_ll]
axes[2].bar(["Base", "Hybrid FT"], ll_vals, color=["#7a7a7a", "#5c80bc"])
axes[2].set_title(f"GSM8K LogLikelihood (n={EVAL_N})")
axes[2].set_ylabel("Mean LL")
for i, v in enumerate(ll_vals):
    axes[2].text(i, v, f"{v:.3f}", ha="center", va="bottom")

plt.suptitle(f"Hybrid Option D v5 Evaluation | Acc Delta: {acc_delta:+.4f}, LL Delta: {ll_delta:+.4f}")
plt.tight_layout()
plt.show()

# Cleanup (optional)
del base_model, base_tokenizer, ft_model, ft_tokenizer
torch.cuda.empty_cache()


## 4.Alte Trainings




In [ ]:
# Config laden und starten
cfg = load_config(f"{PROJECT_DIR}/configs/gsm8k_last_layer.yaml")

# Config anzeigen
import json
print(json.dumps(cfg, indent=2))

In [ ]:
# Optional: Werte überschreiben für schnellen Test mit vollem Layer
cfg["layer_selection"]["layer_indices"] = [23]
cfg["layer_selection"]["components"] = "all"   # alle ~14.9M Params

cfg["es"]["num_generations"] = 10              # Schneller Testlauf
cfg["es"]["population_size"] = 10
cfg["es"]["sigma"] = 0.001                     # Klein weil 14.9M Dimensionen
cfg["es"]["learning_rate"] = 0.001

cfg["fitness"]["num_samples"] = 15             # Samples pro Fitness-Eval
cfg["fitness"]["max_new_tokens"] = 128

cfg["output"]["dir"] = "/content/experiments/runs"

# ACHTUNG: Bei Option A wird das Modell NEU geladen.
# Vorheriges Modell freigeben, um VRAM zu sparen:
del model, tokenizer
torch.cuda.empty_cache()

# Training starten!
run_dir = train(cfg)
print(f"\n✅ Training abgeschlossen! Ergebnisse in: {run_dir}")

### Option B: Manueller Training-Loop (mehr Kontrolle)

Zwei Varianten:
- **B1: Accuracy-Fitness** – generiert Antworten, prüft ob richtig (langsam, diskret)
- **B2: Log-Likelihood-Fitness** – misst Wahrscheinlichkeit der richtigen Antwort (schnell, stetig)

**Empfehlung: B2 (Log-Likelihood)** → ~10× schneller, besseres Signal für ES

In [ ]:
# ── Option B1: Manueller ES-Loop mit ACCURACY-Fitness (langsam, diskret) ──
# (Gleicher Ansatz wie der erste Run → ~10 Min./Generation)

import torch
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k import GSM8KFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen – KOMPLETTER letzter Layer (~14.9M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[22],       # Letzter Decoder-Block
    components="all",         # Attention + MLP + LayerNorm → ~14.9M Params
)
print(f"Target: {selector.num_target_elements:,} Parameter")

# 3) ES-Algorithmus
es = OpenAIES(
    population_size=100,
    sigma=0.002,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
)

# 4) Fitness-Funktion (Accuracy – langsam)
fitness_fn = GSM8KFitness(num_samples=30, split="train", max_new_tokens=256, reshuffle=True, pool_size=3000, seed=42)

# 5) Training
current_params = selector.get_flat_params().clone()
history = []

NUM_GENERATIONS = 30

for gen in range(1, NUM_GENERATIONS + 1):
    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)
        torch.cuda.empty_cache()
    
    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)
    
    history.append({"gen": gen, "best": result.best_fitness, "mean": result.mean_fitness})
    print(f"Gen {gen:3d} | best={result.best_fitness:.3f}  mean={result.mean_fitness:.3f}")

print(f"\n✅ Training abgeschlossen nach {NUM_GENERATIONS} Generationen")

In [ ]:
# ── Option B2: Manueller ES-Loop mit LOG-LIKELIHOOD-Fitness (schnell, stetig) ──
# ~10x schneller als B1 → ~1 Min./Generation statt ~10 Min.

import torch
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswaehlen - KOMPLETTER letzter Layer (~14.9M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[23],
    components="all",
)
print(f"Target: {selector.num_target_elements:,} Parameter")

# 3) ES-Algorithmus (gleiche Settings wie erster Run)
es = OpenAIES(
    population_size=10,
    sigma=0.001,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
)

# 4) Fitness-Funktion (Log-Likelihood - schnell und stetig!)
fitness_fn = GSM8KLogLikelihoodFitness(
    num_samples=30,        # mehr Samples moeglich weil viel schneller
    split="train",
    target_mode="short",   # nur "#### <number>" bewerten
)

# 5) Training
current_params = selector.get_flat_params().clone()
history = []

NUM_GENERATIONS = 10

for gen in range(1, NUM_GENERATIONS + 1):
    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)
        torch.cuda.empty_cache()
    
    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)
    
    history.append({"gen": gen, "best": result.best_fitness, "mean": result.mean_fitness})
    print(f"Gen {gen:3d} | best={result.best_fitness:.4f}  mean={result.mean_fitness:.4f}")

print(f"\nLL-Training abgeschlossen nach {NUM_GENERATIONS} Generationen")

### Option C: Optimiertes ES-Training v3 für A100/H100 (empfohlen)

Basierend auf **Salimans et al. 2017** – "Evolution Strategies as a Scalable Alternative to RL"

**v3-Fixes gegenüber v2:**

| Problem in v2 | Fix in v3 |
|---|---|
| **Reshuffling pro Kandidat** → jeder Kandidat sah andere Fragen, ES-Gradient war Rauschen | Reshuffle pro **Generation**: alle Kandidaten sehen gleiche Fragen |
| Training stagnierte ab Gen 70, lief aber bis 150 | `num_generations: 80` → kein verschwendetes Compute |
| σ sank auf 0.0008 → Exploration tot | `sigma_final: 0.0015` → bleibt lebensfähig |
| Tokenisierung 96× pro Gen (jeder Kandidat neu) | 1× pro Gen → **massiver Speedup** |

**Kernfeatures:**
- **Full-Reasoning Target**: LL über gesamte Step-by-Step Lösung
- **Data Reshuffling per Generation**: Alle 96 Kandidaten der gleichen Gen → gleiche 100 Fragen. Nächste Gen → neue 100 Fragen aus Pool
- **Adam-Optimizer + Cosine σ-Decay**
- **Mittlere Layers [20,21]** statt letzter → kein Catastrophic Forgetting

**Daten-Trennung:** Training auf `train`-Split (7473), Evaluation auf `test`-Split (1319)

**Erwartete Laufzeit:** ~30-45 Min. auf A100

In [ ]:
# ── Option C: Optimiertes ES-Training v3 (Salimans et al. 2017) ──
# BUGFIX: Reshuffle pro Generation (nicht pro Kandidat!)
# Budget: ~30-45 Min. auf A100

import torch, time
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen – Attention von Layer 20+21 (~3.6M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[20, 21],
    components="attention",
)
print(f"Target: {selector.num_target_elements:,} Parameter")
print(selector.summary())

# 3) ES mit Adam-Optimizer + Cosine Sigma-Schedule
NUM_GENERATIONS = 80

es = OpenAIES(
    population_size=96,
    sigma=0.004,                      # Mehr Exploration am Anfang
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
    weight_decay=0.003,               # Weniger aggressiv als v2
    optimizer="adam",
    sigma_decay="cosine",
    sigma_final=0.0015,               # Höher → Exploration bleibt aktiv
    max_generations=NUM_GENERATIONS,
)

# 4) Fitness: Full-Reasoning LL mit per-Generation Reshuffling
#    WICHTIG: reshuffle_data() wird pro Generation aufgerufen (nicht pro Kandidat!)
#    → Alle 96 Kandidaten in einer Gen sehen die GLEICHEN 100 Fragen
#    → Nächste Generation bekommt ANDERE 100 Fragen aus dem Pool
fitness_fn = GSM8KLogLikelihoodFitness(
    num_samples=100,
    split="train",
    target_mode="full",
    batch_size=8,
    reshuffle=True,
    pool_size=5000,
)

# 5) Training Loop
current_params = selector.get_flat_params().clone()
best_ever = -float("inf")
best_ever_params = current_params.clone()
history = []

print(f"\n{'='*65}")
print(f"  ES-Training v3: {NUM_GENERATIONS} Generationen, Pop={es.population_size}")
print(f"  Optimizer: Adam (β1={es.adam_beta1}, β2={es.adam_beta2})")
print(f"  σ-Schedule: cosine {es.sigma:.4f} → {es.sigma_final:.4f}")
print(f"  Target: {selector.num_target_elements:,} Parameter (Layer 20+21 Attention)")
print(f"  Fitness: Full-Reasoning LL (n={fitness_fn.num_samples}, batch={fitness_fn.batch_size})")
print(f"  Data: Reshuffle per GENERATION – 100 gleiche Fragen für alle Kandidaten")
print(f"  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB belegt")
print(f"{'='*65}\n")

t_total = time.time()

for gen in range(1, NUM_GENERATIONS + 1):
    t0 = time.time()

    # WICHTIG: Neue Fragen für diese Generation (alle Kandidaten sehen die gleichen!)
    fitness_fn.reshuffle_data()

    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)

    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)

    if result.best_fitness > best_ever:
        best_ever = result.best_fitness
        best_ever_params = current_params.clone()

    elapsed = time.time() - t0
    history.append({
        "gen": gen, "best": result.best_fitness, "mean": result.mean_fitness,
        "std": result.std_fitness, "best_ever": best_ever,
        "sigma": es.current_sigma, "elapsed": elapsed,
    })

    if gen % 10 == 0 or gen == 1:
        total_t = time.time() - t_total
        eta = (total_t / gen) * (NUM_GENERATIONS - gen)
        print(f"Gen {gen:3d}/{NUM_GENERATIONS} | best={result.best_fitness:.4f}  "
              f"mean={result.mean_fitness:.4f}  best_ever={best_ever:.4f}  "
              f"σ={es.current_sigma:.5f}  [{elapsed:.1f}s/gen, {total_t:.0f}s total, "
              f"ETA ~{eta/60:.0f}min]")

    torch.cuda.empty_cache()

# Best-ever Parameter anwenden
selector.set_flat_params(best_ever_params)
total_time = time.time() - t_total
print(f"\n✅ Training abgeschlossen in {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"   Best-ever LL Fitness: {best_ever:.4f}")

In [ ]:
# ── Vergleich: Fine-tuned (Option C v3) vs Base-Modell auf GSM8K TEST-Split ──
# WICHTIG: Evaluation auf TEST-Split (1319 Fragen) – komplett andere Daten als Training!
# Training nutzte TRAIN-Split (7473 Fragen) → kein Data Leakage

from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.model.loader import load_model_and_tokenizer as load_base
import torch, time

NUM_EVAL = 200  # 200 Fragen aus dem TEST-Split

# --- 1) Fine-tuned Modell evaluieren (aktuell im Speicher) ---
print(f"Evaluiere FINE-TUNED Modell (Option C v3) auf {NUM_EVAL} GSM8K-TEST-Fragen...")
print(f"  (Training war auf TRAIN-Split, Eval ist auf TEST-Split → faire Bewertung)")
eval_fn = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
finetuned_acc = eval_fn.evaluate(model, tokenizer)
t_ft = time.time() - t0
print(f"  Fine-tuned Accuracy: {finetuned_acc:.2%}  ({t_ft:.0f}s)")

# --- 2) Base-Modell laden und evaluieren ---
print(f"\nLade Base-Modell für Vergleich...")
base_model, base_tokenizer = load_base(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"Evaluiere BASE-Modell auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn_base = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
base_acc = eval_fn_base.evaluate(base_model, base_tokenizer)
t_base = time.time() - t0
print(f"  Base Accuracy:       {base_acc:.2%}  ({t_base:.0f}s)")

# Base-Modell freigeben
del base_model, base_tokenizer
torch.cuda.empty_cache()

# --- 3) Ergebnis ---
diff = finetuned_acc - base_acc
print(f"\n{'='*60}")
print(f"  EVALUATION AUF GSM8K TEST-SPLIT (n={NUM_EVAL})")
print(f"  (Training war auf TRAIN-Split → kein Data Leakage)")
print(f"{'='*60}")
print(f"  Base-Modell:      {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Fine-tuned (v3):  {finetuned_acc:6.2%}  ({int(finetuned_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Differenz:        {diff:+6.2%}")
print(f"{'='*60}")

In [ ]:
# ── Visualisierung: Training + Accuracy-Vergleich ──

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

# ── Plot 1: Fitness über Generationen ──
ax1 = fig.add_subplot(gs[0, 0])
gens = [h["gen"] for h in history]
bests = [h["best"] for h in history]
means = [h["mean"] for h in history]
best_evers = [h["best_ever"] for h in history]

ax1.plot(gens, bests, label="Best Fitness", linewidth=1.5, alpha=0.7, color="#2196F3")
ax1.plot(gens, means, label="Mean Fitness", linewidth=1.5, alpha=0.5, color="#FF9800")
ax1.plot(gens, best_evers, label="Best-Ever", linewidth=2, color="#4CAF50", linestyle="--")
ax1.set_xlabel("Generation")
ax1.set_ylabel("Log-Likelihood Fitness")
ax1.set_title("ES Training v3 – Fitness über Generationen\n(Reshuffle-Bugfix + Cosine σ)")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Sigma-Schedule ──
ax2 = fig.add_subplot(gs[0, 1])
sigmas = [h["sigma"] for h in history]
ax2.plot(gens, sigmas, linewidth=2, color="#9C27B0")
ax2.set_xlabel("Generation")
ax2.set_ylabel("σ (Noise Std)")
ax2.set_title("Cosine σ-Decay Schedule")
ax2.grid(True, alpha=0.3)
ax2.fill_between(gens, sigmas, alpha=0.15, color="#9C27B0")

# ── Plot 3: Accuracy-Vergleich (Balkendiagramm) ──
ax3 = fig.add_subplot(gs[1, 0])
labels = ["Base-Modell\n(unverändert)", "Fine-tuned v3\n(ES + Full LL + Reshuffle)"]
accs = [base_acc, finetuned_acc]
colors = ["#4C72B0", "#55A868"]

bars = ax3.bar(labels, accs, color=colors, width=0.5, edgecolor="black", linewidth=0.8)
for bar, acc in zip(bars, accs):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{acc:.1%}\n({int(acc*NUM_EVAL)}/{NUM_EVAL})",
             ha="center", va="bottom", fontweight="bold", fontsize=11)

ax3.set_ylabel("Accuracy")
ax3.set_title(f"GSM8K TEST Accuracy (n={NUM_EVAL})\nTrain auf train-Split, Eval auf test-Split")
ax3.set_ylim(0, max(accs) * 1.3)
ax3.axhline(y=base_acc, color="red", linestyle="--", alpha=0.3, label="Baseline")
ax3.grid(axis="y", alpha=0.3)
ax3.legend(fontsize=9)

# ── Plot 4: Zeitverlauf pro Generation ──
ax4 = fig.add_subplot(gs[1, 1])
times = [h["elapsed"] for h in history]
ax4.plot(gens, times, linewidth=1.5, color="#FF5722", alpha=0.7)
ax4.axhline(y=np.mean(times), color="gray", linestyle="--", alpha=0.5,
            label=f"Ø {np.mean(times):.1f}s/gen")
ax4.set_xlabel("Generation")
ax4.set_ylabel("Zeit (Sekunden)")
ax4.set_title("Laufzeit pro Generation")
ax4.grid(True, alpha=0.3)
ax4.legend(fontsize=9)

fig.suptitle("ES-Training v3 – Cosine σ + Full Reasoning + Reshuffle + Layer 20/21",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Zusammenfassung ──
total_time = sum(times)
print(f"\n{'='*65}")
print(f"  ZUSAMMENFASSUNG – ES-Training v3")
print(f"{'='*65}")
print(f"  Generationen:      {len(history)}")
print(f"  Population:        {es.population_size}")
print(f"  Parameter:         {selector.num_target_elements:,} (Layer 20+21 Attention)")
print(f"  Gesamtzeit:        {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"  Ø pro Generation:  {np.mean(times):.1f}s")
print(f"  σ Start → Ende:   {history[0]['sigma']:.5f} → {history[-1]['sigma']:.5f}")
print(f"  Best-Ever LL:      {history[-1]['best_ever']:.4f}")
print(f"  Data Reshuffling:  ON (100 neue Fragen/Gen aus Pool von 5000)")
print(f"  Target Mode:       full (gesamtes Reasoning)")

### Option D: ES-Training v5 – Breitere Layer-Abdeckung (empfohlen nach v3)

Aufbauend auf v3 (+5%) — doppelt so viele Layers für mehr Kapazität.

| | v3 | v5 |
|---|---|---|
| **Layers** | [20, 21] ~3.6M Params | [18, 19, 20, 21] ~7.2M Params |
| **σ** | 0.004 → 0.0015 | 0.003 → 0.001 (kleiner: mehr Params) |
| **LR** | 0.001 | 0.0008 (konservativer) |
| **Generationen** | 80 | 100 |
| **Samples/Gen** | 100 | 120 |
| **Batch-Size** | 8 | 12 (A100 hat genug VRAM) |
| **Laufzeit** | ~60 min | ~75 min |

**Warum 4 Layers?** v3 mit 2 Layers hat +5% erreicht — möglicherweise Kapazitäts-Limit. Layers 18+19 sind sicher (kein Catastrophic Forgetting).

In [ ]:
# ── Option D: ES-Training v5 – 4 Layers (Salimans et al. 2017) ──
# Doppelte Layer-Abdeckung: [18,19,20,21] → ~7.2M Parameter
# Budget: ~75 Min. auf A100

import torch, time
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness

# 1) Modell laden
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen – Attention von Layer 18-21 (~7.2M Parameter)
selector = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[18, 19, 20, 21],
    components="attention",
)
print(f"Target: {selector.num_target_elements:,} Parameter")
print(selector.summary())

# 3) ES mit Adam-Optimizer + Cosine Sigma-Schedule
NUM_GENERATIONS = 100

es = OpenAIES(
    population_size=96,
    sigma=0.003,                      # Kleiner als v3: mehr Params braucht weniger Noise
    learning_rate=0.0008,             # Konservativer bei mehr Params
    antithetic=True,
    fitness_shaping="centered_rank",
    weight_decay=0.002,               # Leichter: mehr Params braucht weniger Druck
    optimizer="adam",
    sigma_decay="cosine",
    sigma_final=0.001,                # Kann tiefer → mehr Kapazität zum Exploitieren
    max_generations=NUM_GENERATIONS,
)

# 4) Fitness: Full-Reasoning LL mit per-Generation Reshuffling
fitness_fn = GSM8KLogLikelihoodFitness(
    num_samples=120,                  # Mehr Samples → stabilerer Gradient
    split="train",
    target_mode="full",
    batch_size=12,                    # A100: 12 passt locker bei 0.5B-Modell
    reshuffle=True,
    pool_size=5000,
)

# 5) Training Loop
current_params = selector.get_flat_params().clone()
best_ever = -float("inf")
best_ever_params = current_params.clone()
history = []

print(f"\n{'='*65}")
print(f"  ES-Training v5: {NUM_GENERATIONS} Generationen, Pop={es.population_size}")
print(f"  Optimizer: Adam (β1={es.adam_beta1}, β2={es.adam_beta2})")
print(f"  σ-Schedule: cosine {es.sigma:.4f} → {es.sigma_final:.4f}")
print(f"  Target: {selector.num_target_elements:,} Parameter (Layer 18-21 Attention)")
print(f"  Fitness: Full-Reasoning LL (n={fitness_fn.num_samples}, batch={fitness_fn.batch_size})")
print(f"  Data: Reshuffle per GENERATION – 120 gleiche Fragen für alle Kandidaten")
print(f"  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB belegt")
print(f"{'='*65}\n")

t_total = time.time()

for gen in range(1, NUM_GENERATIONS + 1):
    t0 = time.time()

    # Neue Fragen für diese Generation
    fitness_fn.reshuffle_data()

    candidates = es.ask(current_params)
    fitnesses = []
    for cand in candidates:
        selector.set_flat_params(cand)
        fit = fitness_fn.evaluate(model, tokenizer)
        fitnesses.append(fit)

    result = es.tell(current_params, candidates, fitnesses)
    current_params = result.new_params.clone()
    selector.set_flat_params(current_params)

    if result.best_fitness > best_ever:
        best_ever = result.best_fitness
        best_ever_params = current_params.clone()

    elapsed = time.time() - t0
    history.append({
        "gen": gen, "best": result.best_fitness, "mean": result.mean_fitness,
        "std": result.std_fitness, "best_ever": best_ever,
        "sigma": es.current_sigma, "elapsed": elapsed,
    })

    if gen % 10 == 0 or gen == 1:
        total_t = time.time() - t_total
        eta = (total_t / gen) * (NUM_GENERATIONS - gen)
        print(f"Gen {gen:3d}/{NUM_GENERATIONS} | best={result.best_fitness:.4f}  "
              f"mean={result.mean_fitness:.4f}  best_ever={best_ever:.4f}  "
              f"σ={es.current_sigma:.5f}  [{elapsed:.1f}s/gen, {total_t:.0f}s total, "
              f"ETA ~{eta/60:.0f}min]")

    torch.cuda.empty_cache()

# Best-ever Parameter anwenden
selector.set_flat_params(best_ever_params)
total_time = time.time() - t_total
print(f"\n✅ Training abgeschlossen in {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"   Best-ever LL Fitness: {best_ever:.4f}")

In [ ]:
# ── Option D v5: OLMES-Test (GSM8K + HellaSwag) ──

import json
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

# 1) Fine-tuned Modell aus Option D speichern
run_tag_eval = datetime.now().strftime("%Y%m%d_%H%M%S")
output_base = Path(globals().get("OUTPUT_BASE", "/content/experiments"))
D5_OUT = output_base / f"option_d5_olmes_{run_tag_eval}"
D5_MODEL_DIR = D5_OUT / "model"
D5_MODEL_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(D5_MODEL_DIR))
tokenizer.save_pretrained(str(D5_MODEL_DIR))
print("Saved Option D v5 model:", D5_MODEL_DIR)


def run_cmd(cmd, check=True, cwd=None):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, text=True, capture_output=True, cwd=cwd)
    if p.stdout:
        print(p.stdout[-1200:])
    if p.returncode != 0 and p.stderr:
        print(p.stderr[-1200:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
    return p


def ensure_olmes_ready():
    if shutil.which("olmes") or shutil.which("oe-eval"):
        return

    olmes_dir = Path("/content/olmes")
    if not olmes_dir.exists():
        run_cmd(["git", "clone", "https://github.com/allenai/olmes.git", str(olmes_dir)])
    else:
        run_cmd(["git", "-C", str(olmes_dir), "pull"])

    run_cmd(["python3", "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
    run_cmd(["python3", "-m", "pip", "install", "-q", "-e", "."], cwd=str(olmes_dir))
    run_cmd(["python3", "-m", "pip", "install", "-q", "-e", ".[gpu]"], check=False, cwd=str(olmes_dir))


ensure_olmes_ready()

# 2) OLMES-Eval via bestehendem Wrapper
script_path = Path("scripts/eval_olmes_tasks.py")
if not script_path.exists():
    raise FileNotFoundError("scripts/eval_olmes_tasks.py nicht gefunden.")

EVAL_TASKS = [
    ("gsm8k::olmes", "gsm8k"),
    ("hellaswag::olmo1", "hellaswag"),
]

rows = []
for task_name, short_name in EVAL_TASKS:
    out_dir = D5_OUT / "olmes_eval" / short_name
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python3", str(script_path),
        "--model", str(D5_MODEL_DIR),
        "--tasks", task_name,
        "--model_type", "hf",
        "--limit", "200",
        "--num_shots", "4",
        "--batch_size", "8",
        "--cache_dir", "/content/cache",
        "--output_dir", str(out_dir),
    ]
    run_cmd(cmd, check=True)

    summary_path = out_dir / "run_summary.json"
    metrics_path = out_dir / "metrics.json"
    row = {
        "task": task_name,
        "output_dir": str(out_dir),
        "run_summary_exists": summary_path.exists(),
        "metrics_exists": metrics_path.exists(),
    }
    if summary_path.exists():
        row.update(json.loads(summary_path.read_text(encoding="utf-8")))
    rows.append(row)

print("\nOption D v5 OLMES outputs:", D5_OUT / "olmes_eval")
print(json.dumps(rows, indent=2))

globals()["D5_OLMES_OUT"] = D5_OUT


### Option D v5 OLMES: Eval auf trainiertem Modell (GSM8K + HellaSwag)

Dieser Block ist explizit für **Option D v5** (`model`, `tokenizer`) gedacht:
- Speichert das trainierte Option-D-Modell
- Führt OLMES auf `gsm8k::olmes` und `hellaswag::olmo1` aus
- Erstellt zusammengefasste Metriken und Visualisierung

In [ ]:
# 4d) OLMES Eval: Option D v5 Fine-tuned Modell auf GSM8K + HellaSwag

import json
import subprocess
from datetime import datetime
from pathlib import Path

# ---- Preconditions ----
if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "Bitte zuerst Option D v5 Training ausfuehren (die Zelle mit model/tokenizer), "
        "damit das fine-tuned Modell im Speicher ist."
    )

# ---- Standalone Defaults ----
output_base = globals().get("OUTPUT_BASE", Path("/content/experiments/olmes"))
run_tag = globals().get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
if not isinstance(output_base, Path):
    output_base = Path(output_base)

D5_OLMES_FT_OUT = output_base / f"olmes_option_d5_finetuned_gsm8k_hellaswag_{run_tag}"
D5_OLMES_FT_MODEL = D5_OLMES_FT_OUT / "model"
D5_OLMES_FT_GSM8K = D5_OLMES_FT_OUT / "gsm8k"
D5_OLMES_FT_HELLA = D5_OLMES_FT_OUT / "hellaswag"

for p in [D5_OLMES_FT_OUT, D5_OLMES_FT_MODEL, D5_OLMES_FT_GSM8K, D5_OLMES_FT_HELLA]:
    p.mkdir(parents=True, exist_ok=True)

globals()["D5_OLMES_FT_OUT"] = D5_OLMES_FT_OUT
globals()["D5_OLMES_FT_MODEL"] = D5_OLMES_FT_MODEL
globals()["D5_OLMES_FT_GSM8K"] = D5_OLMES_FT_GSM8K
globals()["D5_OLMES_FT_HELLA"] = D5_OLMES_FT_HELLA

# ---- Save fine-tuned Option D model ----
model.save_pretrained(str(D5_OLMES_FT_MODEL))
tokenizer.save_pretrained(str(D5_OLMES_FT_MODEL))
print("Saved Option D v5 model:", D5_OLMES_FT_MODEL)


def run(cmd, check=True, cwd=None):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, text=True, capture_output=True, cwd=cwd)
    if p.stdout:
        print(p.stdout[-1500:])
    if p.returncode != 0 and p.stderr:
        print(p.stderr[-1500:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
    return p

# ---- 1) OLMES bereitstellen ----
OLMES_DIR = Path("/content/olmes")
if not OLMES_DIR.exists():
    run(["git", "clone", "https://github.com/allenai/olmes.git", str(OLMES_DIR)])
else:
    run(["git", "-C", str(OLMES_DIR), "pull"])

# ---- 2) Installation ----
run(["python3", "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
run(["python3", "-m", "pip", "install", "-q", "-e", "."], cwd=str(OLMES_DIR))
gpu_install = run(["python3", "-m", "pip", "install", "-q", "-e", ".[gpu]"], check=False, cwd=str(OLMES_DIR))
if gpu_install.returncode == 0:
    print("OLMES GPU extras installiert (vLLM verfuegbar).")
else:
    print("Hinweis: .[gpu] konnte nicht installiert werden. Nutze --model_type hf.")

# ---- 3) Eval je Task ----
script_path = Path("scripts/eval_olmes_tasks.py")
if not script_path.exists():
    raise FileNotFoundError("scripts/eval_olmes_tasks.py nicht gefunden.")

task_runs = [
    ("gsm8k::olmes", D5_OLMES_FT_GSM8K),
    ("hellaswag::olmo1", D5_OLMES_FT_HELLA),
]

for task_name, task_out in task_runs:
    cmd = [
        "python3", str(script_path),
        "--model", str(D5_OLMES_FT_MODEL),
        "--tasks", task_name,
        "--model_type", "hf",
        "--limit", "200",
        "--num_shots", "0",
        "--batch_size", "8",
        "--cache_dir", "/content/cache",
        "--output_dir", str(task_out),
    ]
    run(cmd, check=True)

summary = {
    "model": str(D5_OLMES_FT_MODEL),
    "output_root": str(D5_OLMES_FT_OUT),
    "task_outputs": {
        "gsm8k::olmes": str(D5_OLMES_FT_GSM8K),
        "hellaswag::olmo1": str(D5_OLMES_FT_HELLA),
    },
}
(D5_OLMES_FT_OUT / "combined_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("✅ Option D v5 fine-tuned OLMES outputs:")
print("- GSM8K:", D5_OLMES_FT_GSM8K)
print("- HellaSwag:", D5_OLMES_FT_HELLA)

In [ ]:
# 4e) Visualisierung: Option D v5 Fine-tuned OLMES (GSM8K + HellaSwag)

import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def _read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _extract_task_entries(metrics_obj):
    if isinstance(metrics_obj.get("tasks"), list):
        return metrics_obj["tasks"]
    if isinstance(metrics_obj.get("tasks"), dict):
        out = []
        for name, val in metrics_obj["tasks"].items():
            if isinstance(val, dict):
                e = dict(val)
                e.setdefault("task_name", name)
                out.append(e)
        return out
    return []


def _normalize_task_name(raw_name: str) -> str:
    return (raw_name or "").strip().lower().replace(" ", "")


def _pick_accuracy_metric(task_name: str, row: dict):
    t = _normalize_task_name(task_name)
    if "gsm8k" in t:
        for key in ["exact_match", "exact_match_flex", "primary_score", "acc_norm", "acc_raw"]:
            val = row.get(key)
            if val is not None and pd.notna(val):
                return key, float(val)
    if "hellaswag" in t:
        for key in ["acc_norm", "acc_raw", "primary_score", "exact_match", "exact_match_flex"]:
            val = row.get(key)
            if val is not None and pd.notna(val):
                return key, float(val)
    for key in ["primary_score", "acc_norm", "acc_raw", "exact_match", "exact_match_flex"]:
        val = row.get(key)
        if val is not None and pd.notna(val):
            return key, float(val)
    return None, None


def _resolve_output_root() -> Path:
    root = globals().get("D5_OLMES_FT_OUT")
    if root is not None and (Path(root) / "combined_summary.json").exists():
        return Path(root)

    base = Path(globals().get("OUTPUT_BASE", Path.cwd() / "experiments" / "olmes"))
    candidates = sorted(base.glob("olmes_option_d5_finetuned_gsm8k_hellaswag_*"), key=lambda p: p.stat().st_mtime, reverse=True)
    for c in candidates:
        if (c / "combined_summary.json").exists():
            globals()["D5_OLMES_FT_OUT"] = c
            return c
    raise FileNotFoundError("Kein combined_summary.json gefunden. Fuehre zuerst die OLMES-Eval-Zelle aus.")


def _resolve_task_output(root: Path, task_key: str, fallback_subdir: str) -> Path:
    summary_path = root / "combined_summary.json"
    if summary_path.exists():
        summary = _read_json(summary_path)
        task_outputs = summary.get("task_outputs", {})
        p = task_outputs.get(task_key)
        if p and (Path(p) / "metrics.json").exists():
            return Path(p)

    fallback = root / fallback_subdir
    if (fallback / "metrics.json").exists():
        return fallback
    raise FileNotFoundError(f"metrics.json fuer {task_key} nicht gefunden unter {fallback}")


def _task_row_from_metrics(metrics_path: Path, expected_task: str):
    metrics = _read_json(metrics_path)
    entries = _extract_task_entries(metrics)

    if not entries:
        return {
            "task": expected_task,
            "task_norm": _normalize_task_name(expected_task),
            "accuracy_metric": None,
            "accuracy": None,
            "source": str(metrics_path),
        }

    chosen = None
    for e in entries:
        name = e.get("alias") or e.get("task_name") or e.get("name") or e.get("id") or "unknown_task"
        if expected_task.split("::")[0] in _normalize_task_name(name):
            chosen = e
            break
    if chosen is None:
        chosen = entries[0]

    m = chosen.get("metrics", {}) or {}
    task_name = chosen.get("alias") or chosen.get("task_name") or chosen.get("name") or chosen.get("id") or expected_task
    row = {
        "task": task_name,
        "task_norm": _normalize_task_name(task_name),
        "primary_score": m.get("primary_score", chosen.get("primary_score")),
        "acc_raw": m.get("acc_raw", chosen.get("acc_raw")),
        "acc_norm": m.get("acc_norm", chosen.get("acc_norm")),
        "exact_match": m.get("exact_match", chosen.get("exact_match")),
        "exact_match_flex": m.get("exact_match_flex", chosen.get("exact_match_flex")),
        "num_instances": chosen.get("num_instances"),
        "processing_time_sec": chosen.get("processing_time"),
        "source": str(metrics_path),
    }
    metric_name, metric_val = _pick_accuracy_metric(task_name, row)
    row["accuracy_metric"] = metric_name
    row["accuracy"] = metric_val
    return row


root = _resolve_output_root()
gsm_metrics = _resolve_task_output(root, "gsm8k::olmes", "gsm8k") / "metrics.json"
hsw_metrics = _resolve_task_output(root, "hellaswag::olmo1", "hellaswag") / "metrics.json"

rows = [
    _task_row_from_metrics(gsm_metrics, "gsm8k::olmes"),
    _task_row_from_metrics(hsw_metrics, "hellaswag::olmo1"),
]
d5_ft_df = pd.DataFrame(rows)
display(d5_ft_df)

plot_df = d5_ft_df.dropna(subset=["accuracy"]).copy()
if plot_df.empty:
    print("Keine plotbaren Accuracy-Werte gefunden.")
else:
    plt.figure(figsize=(9, 4))
    plt.bar(plot_df["task"], plot_df["accuracy"])
    plt.title("Option D v5 Fine-tuned (OLMES): GSM8K vs HellaSwag")
    plt.ylabel("accuracy")
    plt.xlabel("task")
    plt.ylim(0, 1)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

gsm = d5_ft_df[d5_ft_df["task_norm"].str.contains("gsm8k", na=False)]
hsw = d5_ft_df[d5_ft_df["task_norm"].str.contains("hellaswag", na=False)]

if not gsm.empty and not hsw.empty and pd.notna(gsm.iloc[0]["accuracy"]) and pd.notna(hsw.iloc[0]["accuracy"]):
    gsm_row = gsm.iloc[0]
    hsw_row = hsw.iloc[0]
    print("\nAccuracy-Vergleich (Option D v5 fine-tuned):")
    print(f"- GSM8K ({gsm_row['accuracy_metric']}): {gsm_row['accuracy']:.4f}")
    print(f"- HellaSwag ({hsw_row['accuracy_metric']}): {hsw_row['accuracy']:.4f}")
    print(f"- Differenz (HellaSwag - GSM8K): {hsw_row['accuracy'] - gsm_row['accuracy']:+.4f}")
else:
    print("Vergleich unvollstaendig: Mindestens ein Task ohne valide Accuracy.")
    print("GSM8K metrics:", gsm_metrics)
    print("HellaSwag metrics:", hsw_metrics)

print("Option D v5 fine-tuned OLMES output root:", root)
print("- GSM8K metrics:", gsm_metrics)
print("- HellaSwag metrics:", hsw_metrics)

In [ ]:
# ── Vergleich: Fine-tuned (Option D v5) vs Base-Modell auf GSM8K TEST-Split ──

from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.model.loader import load_model_and_tokenizer as load_base
import torch, time

NUM_EVAL = 200

# --- 1) Fine-tuned Modell evaluieren ---
print(f"Evaluiere FINE-TUNED Modell (Option D v5) auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
finetuned_acc = eval_fn.evaluate(model, tokenizer)
t_ft = time.time() - t0
print(f"  Fine-tuned Accuracy: {finetuned_acc:.2%}  ({t_ft:.0f}s)")

# --- 2) Base-Modell laden und evaluieren ---
print(f"\nLade Base-Modell für Vergleich...")
base_model, base_tokenizer = load_base(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"Evaluiere BASE-Modell auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn_base = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
base_acc = eval_fn_base.evaluate(base_model, base_tokenizer)
t_base = time.time() - t0
print(f"  Base Accuracy:       {base_acc:.2%}  ({t_base:.0f}s)")

del base_model, base_tokenizer
torch.cuda.empty_cache()

# --- 3) Ergebnis ---
diff = finetuned_acc - base_acc
print(f"\n{'='*60}")
print(f"  EVALUATION AUF GSM8K TEST-SPLIT (n={NUM_EVAL})")
print(f"{'='*60}")
print(f"  Base-Modell:      {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Fine-tuned (v5):  {finetuned_acc:6.2%}  ({int(finetuned_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Differenz:        {diff:+6.2%}")
print(f"{'='*60}")

### Option D-HS: ES-Training auf HellaSwag (Log-Likelihood Multiple Choice)

Trainingsversuch analog zu Option D, aber auf **HellaSwag** statt GSM8K.

- Modell/Layer: Qwen2.5-0.5B, Attention in Layer `[18,19,20,21]`
- Optimierer: OpenAI-ES (Salimans et al. 2017), Adam, cosine sigma schedule
- Fitness: mittlere **Choice-Margin** auf HellaSwag-Train-Subset
- Danach: Evaluation auf separatem Val-Subset + Visualisierung von Trainingskurven

In [ ]:
# ── Option D-HS: Setup + Fitnessklasse fuer HellaSwag ──

import time
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch import nn
from transformers import PreTrainedTokenizerBase

from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES


class HellaSwagLogLikelihoodFitness:
    """ES fitness on HellaSwag using log-likelihood over answer choices.

    For each example, compute LL for each of 4 endings and use:
      margin = LL(correct) - max(LL(other choices))
    Fitness = mean margin over sampled examples.
    """

    SYSTEM_PROMPT = "You are a helpful assistant."

    def __init__(
        self,
        num_samples: int = 48,
        split: str = "train",
        seed: int = 42,
        reshuffle: bool = True,
        pool_size: int | None = 10000,
        max_length: int = 512,
    ):
        self.num_samples = num_samples
        self.split = split
        self.seed = seed
        self.reshuffle = reshuffle
        self.pool_size = pool_size
        self.max_length = max_length

        self._rng = random.Random(seed)
        self._full_pool = None
        self._dataset = None
        self._needs_reshuffle = True

    def reshuffle_data(self):
        if self.reshuffle:
            self._needs_reshuffle = True

    def _load_data(self):
        if self._full_pool is None:
            ds = load_dataset("hellaswag", split=self.split)
            ds = ds.shuffle(seed=self.seed)
            if self.pool_size is not None:
                ds = ds.select(range(min(self.pool_size, len(ds))))
            self._full_pool = ds
            print(f"HellaSwag pool loaded: {len(self._full_pool)} samples ({self.split})")

        if self.reshuffle and self._needs_reshuffle:
            idx = self._rng.sample(
                range(len(self._full_pool)),
                min(self.num_samples, len(self._full_pool)),
            )
            self._dataset = self._full_pool.select(idx)
            self._needs_reshuffle = False
        elif self._dataset is None:
            self._dataset = self._full_pool.select(range(min(self.num_samples, len(self._full_pool))))

    def _build_prompt(self, ex: dict, tokenizer: PreTrainedTokenizerBase) -> str:
        ctx = ex["ctx"].strip()
        user = (
            "Complete the following context with the most plausible ending.\n"
            "Context:\n"
            f"{ctx}\n"
            "Answer:" 
        )
        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    @torch.inference_mode()
    def _choice_ll(
        self,
        model: nn.Module,
        tokenizer: PreTrainedTokenizerBase,
        prompt_text: str,
        ending: str,
    ) -> float:
        device = next(model.parameters()).device

        full_text = prompt_text + " " + ending.strip()
        prompt_ids = tokenizer(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
        )["input_ids"][0]
        full_ids = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
        )["input_ids"][0]

        if full_ids.shape[0] <= prompt_ids.shape[0]:
            return -1e9

        input_ids = full_ids.unsqueeze(0).to(device)
        attn = torch.ones_like(input_ids, device=device)

        logits = model(input_ids, attention_mask=attn).logits
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

        ans_start = prompt_ids.shape[0]
        seq_len = full_ids.shape[0]

        ans_log_probs = log_probs[0, ans_start - 1 : seq_len - 1, :]
        ans_tokens = input_ids[0, ans_start:seq_len]
        token_lls = ans_log_probs.gather(1, ans_tokens.unsqueeze(1)).squeeze(1)

        return token_lls.mean().item()

    @torch.inference_mode()
    def evaluate(self, model: nn.Module, tokenizer: PreTrainedTokenizerBase) -> float:
        self._load_data()
        model.eval()

        margins = []
        for ex in self._dataset:
            prompt = self._build_prompt(ex, tokenizer)
            endings = ex["endings"]
            gold = int(ex["label"])

            lls = [self._choice_ll(model, tokenizer, prompt, c) for c in endings]
            gold_ll = lls[gold]
            best_other = max([ll for i, ll in enumerate(lls) if i != gold])
            margins.append(gold_ll - best_other)

        if not margins:
            return -float("inf")
        return float(np.mean(margins))


@torch.inference_mode()
def eval_hellaswag_accuracy(
    model: nn.Module,
    tokenizer: PreTrainedTokenizerBase,
    split: str = "validation",
    n: int = 200,
    seed: int = 42,
    max_length: int = 512,
):
    ds = load_dataset("hellaswag", split=split).shuffle(seed=seed)
    ds = ds.select(range(min(n, len(ds))))

    fitness = HellaSwagLogLikelihoodFitness(
        num_samples=min(n, len(ds)),
        split=split,
        seed=seed,
        reshuffle=False,
        pool_size=n,
        max_length=max_length,
    )
    fitness._full_pool = ds
    fitness._dataset = ds

    correct = 0
    for ex in ds:
        prompt = fitness._build_prompt(ex, tokenizer)
        endings = ex["endings"]
        gold = int(ex["label"])

        lls = [fitness._choice_ll(model, tokenizer, prompt, c) for c in endings]
        pred = int(np.argmax(lls))
        correct += int(pred == gold)

    return correct / len(ds) if len(ds) > 0 else 0.0

In [ ]:
# ── Option D-HS: Evaluation (HellaSwag validation) ──

HS_EVAL_N = 200

print(f"Evaluate fine-tuned HellaSwag model on validation (n={HS_EVAL_N})...")
t0 = time.time()
ft_hs_acc = eval_hellaswag_accuracy(
    model_hs,
    tokenizer_hs,
    split="validation",
    n=HS_EVAL_N,
    seed=123,
    max_length=512,
)
ft_eval_time = time.time() - t0
print(f"Fine-tuned HellaSwag accuracy: {ft_hs_acc:.2%} ({ft_eval_time:.1f}s)")

print("\nLoad base model for fair comparison...")
base_model_hs, base_tokenizer_hs = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"Evaluate base model on same validation subset (n={HS_EVAL_N})...")
t0 = time.time()
base_hs_acc = eval_hellaswag_accuracy(
    base_model_hs,
    base_tokenizer_hs,
    split="validation",
    n=HS_EVAL_N,
    seed=123,
    max_length=512,
)
base_eval_time = time.time() - t0
print(f"Base HellaSwag accuracy:       {base_hs_acc:.2%} ({base_eval_time:.1f}s)")

delta_hs = ft_hs_acc - base_hs_acc
print(f"\nDelta (fine-tuned - base): {delta_hs:+.2%}")

# cleanup
try:
    del base_model_hs, base_tokenizer_hs
    torch.cuda.empty_cache()
except Exception:
    pass

hs_eval_summary = {
    "n": HS_EVAL_N,
    "finetuned_acc": ft_hs_acc,
    "base_acc": base_hs_acc,
    "delta": delta_hs,
    "train_minutes": train_time_hs / 60 if "train_time_hs" in globals() else None,
}
print("\nHellaSwag eval summary:", hs_eval_summary)

In [ ]:
# ── Option D-HS: Visualisierung ──

import pandas as pd

if len(history_hs) == 0:
    raise RuntimeError("history_hs ist leer. Bitte zuerst die Trainingszelle ausfuehren.")

hist_df = pd.DataFrame(history_hs)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1) Fitness-Verlauf
axes[0].plot(hist_df["gen"], hist_df["best"], label="best", lw=2)
axes[0].plot(hist_df["gen"], hist_df["mean"], label="mean", lw=2)
axes[0].plot(hist_df["gen"], hist_df["best_ever"], label="best_ever", lw=2)
axes[0].set_title("Option D-HS: Fitness over Generations")
axes[0].set_xlabel("Generation")
axes[0].set_ylabel("HellaSwag margin fitness")
axes[0].legend()
axes[0].grid(alpha=0.25)

# 2) Sigma + Zeit
ax2 = axes[1]
ax2.plot(hist_df["gen"], hist_df["sigma"], color="purple", lw=2, label="sigma")
ax2.set_title("Sigma schedule + runtime")
ax2.set_xlabel("Generation")
ax2.set_ylabel("Sigma", color="purple")
ax2.tick_params(axis="y", labelcolor="purple")
ax2.grid(alpha=0.25)

ax2b = ax2.twinx()
ax2b.plot(hist_df["gen"], hist_df["elapsed"], color="orange", lw=1.8, alpha=0.75, label="sec/gen")
ax2b.set_ylabel("Seconds per generation", color="orange")
ax2b.tick_params(axis="y", labelcolor="orange")

# 3) Base vs Fine-tuned accuracy
labels = ["Base", "Fine-tuned D-HS"]
vals = [base_hs_acc, ft_hs_acc]
colors = ["gray", "seagreen"]
bars = axes[2].bar(labels, vals, color=colors)
axes[2].set_ylim(0, max(0.4, max(vals) + 0.05))
axes[2].set_title(f"HellaSwag validation accuracy (n={HS_EVAL_N})")
axes[2].set_ylabel("Accuracy")
axes[2].grid(axis="y", alpha=0.25)
for b, v in zip(bars, vals):
    axes[2].text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.2%}", ha="center")

plt.tight_layout()
plt.show()

print("\nSummary")
print("- Best-ever fitness:", f"{hist_df['best_ever'].iloc[-1]:.4f}")
print("- Mean sec/gen:", f"{hist_df['elapsed'].mean():.1f}")
print("- Base acc:", f"{base_hs_acc:.2%}")
print("- Fine-tuned acc:", f"{ft_hs_acc:.2%}")
print("- Delta:", f"{(ft_hs_acc - base_hs_acc):+.2%}")

### Option D-HS OLMES: Eval auf fertig trainiertem HellaSwag-Modell

Analog zur Baseline, aber mit dem **fine-tuned HellaSwag-Modell** (`model_hs`/`tokenizer_hs`).

- Speichert das trainierte Modell lokal
- Führt OLMES separat auf `gsm8k::olmes` und `hellaswag::olmo1` aus
- Schreibt `combined_summary.json`

In [ ]:
# 4d) OLMES Eval: Fine-tuned HellaSwag-Modell auf GSM8K + HellaSwag

import json
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

# ---- Preconditions ----
if "model_hs" not in globals() or "tokenizer_hs" not in globals():
    raise RuntimeError(
        "Bitte zuerst den HellaSwag-Trainingsblock ausfuehren, "
        "damit model_hs/tokenizer_hs vorhanden sind."
    )

# ---- Standalone Defaults ----
output_base = globals().get("OUTPUT_BASE", Path("/content/experiments/olmes"))
run_tag = globals().get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
if not isinstance(output_base, Path):
    output_base = Path(output_base)

FT_HS_OLMES_OUT = output_base / f"olmes_finetuned_hs_gsm8k_hellaswag_{run_tag}"
FT_HS_MODEL_DIR = FT_HS_OLMES_OUT / "model"
FT_HS_GSM8K_OUT = FT_HS_OLMES_OUT / "gsm8k"
FT_HS_HELLASWAG_OUT = FT_HS_OLMES_OUT / "hellaswag"

for p in [FT_HS_OLMES_OUT, FT_HS_MODEL_DIR, FT_HS_GSM8K_OUT, FT_HS_HELLASWAG_OUT]:
    p.mkdir(parents=True, exist_ok=True)

globals()["FT_HS_OLMES_OUT"] = FT_HS_OLMES_OUT
globals()["FT_HS_MODEL_DIR"] = FT_HS_MODEL_DIR
globals()["FT_HS_GSM8K_OUT"] = FT_HS_GSM8K_OUT
globals()["FT_HS_HELLASWAG_OUT"] = FT_HS_HELLASWAG_OUT

# ---- Save fine-tuned model ----
model_hs.save_pretrained(str(FT_HS_MODEL_DIR))
tokenizer_hs.save_pretrained(str(FT_HS_MODEL_DIR))

print(f"Fine-tuned model saved to: {FT_HS_MODEL_DIR}")
print(f"Output root: {FT_HS_OLMES_OUT}")
print(f"GSM8K out:   {FT_HS_GSM8K_OUT}")
print(f"HellaSwag out: {FT_HS_HELLASWAG_OUT}")


def run(cmd, check=True, cwd=None):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, text=True, capture_output=True, cwd=cwd)
    if p.stdout:
        print(p.stdout[-1500:])
    if p.returncode != 0 and p.stderr:
        print(p.stderr[-1500:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
    return p


# ---- 1) Offizielles OLMES klonen/aktualisieren ----
OLMES_DIR = Path("/content/olmes")
if not OLMES_DIR.exists():
    run(["git", "clone", "https://github.com/allenai/olmes.git", str(OLMES_DIR)])
else:
    run(["git", "-C", str(OLMES_DIR), "pull"])

# ---- 2) Installation ----
run(["python3", "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
run(["python3", "-m", "pip", "install", "-q", "-e", "."], cwd=str(OLMES_DIR))
gpu_install = run(["python3", "-m", "pip", "install", "-q", "-e", ".[gpu]"], check=False, cwd=str(OLMES_DIR))
if gpu_install.returncode == 0:
    print("OLMES GPU extras installiert (vLLM verfuegbar).")
else:
    print("Hinweis: .[gpu] konnte nicht installiert werden. Nutze --model_type hf.")

# ---- 3) Fine-tuned Eval: getrennte Runs je Task ----
script_path = Path("scripts/eval_olmes_tasks.py")
if not script_path.exists():
    raise FileNotFoundError("scripts/eval_olmes_tasks.py nicht gefunden.")

task_runs = [
    ("gsm8k::olmes", FT_HS_GSM8K_OUT),
    ("hellaswag::olmo1", FT_HS_HELLASWAG_OUT),
]

for task_name, task_out in task_runs:
    cmd = [
        "python3", str(script_path),
        "--model", str(FT_HS_MODEL_DIR),
        "--tasks", task_name,
        "--model_type", "hf",
        "--limit", "200",
        "--num_shots", "0",
        "--batch_size", "8",
        "--cache_dir", "/content/cache",
        "--output_dir", str(task_out),
    ]
    run(cmd, check=True)

summary = {
    "model": str(FT_HS_MODEL_DIR),
    "output_root": str(FT_HS_OLMES_OUT),
    "task_outputs": {
        "gsm8k::olmes": str(FT_HS_GSM8K_OUT),
        "hellaswag::olmo1": str(FT_HS_HELLASWAG_OUT),
    },
}
(FT_HS_OLMES_OUT / "combined_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("✅ Fine-tuned HellaSwag OLMES outputs:")
print("- GSM8K:", FT_HS_GSM8K_OUT)
print("- HellaSwag:", FT_HS_HELLASWAG_OUT)

In [ ]:
# 4e) Visualisierung: Fine-tuned HellaSwag-Modell (OLMES) GSM8K + HellaSwag

import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def _read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _extract_task_entries(metrics_obj):
    if isinstance(metrics_obj.get("tasks"), list):
        return metrics_obj["tasks"]
    if isinstance(metrics_obj.get("tasks"), dict):
        out = []
        for name, val in metrics_obj["tasks"].items():
            if isinstance(val, dict):
                e = dict(val)
                e.setdefault("task_name", name)
                out.append(e)
        return out
    return []


def _normalize_task_name(raw_name: str) -> str:
    return (raw_name or "").strip().lower().replace(" ", "")


def _pick_accuracy_metric(task_name: str, row: dict):
    t = _normalize_task_name(task_name)
    if "gsm8k" in t:
        for key in ["exact_match", "exact_match_flex", "primary_score", "acc_norm", "acc_raw"]:
            val = row.get(key)
            if val is not None and pd.notna(val):
                return key, float(val)
    if "hellaswag" in t:
        for key in ["acc_norm", "acc_raw", "primary_score", "exact_match", "exact_match_flex"]:
            val = row.get(key)
            if val is not None and pd.notna(val):
                return key, float(val)
    for key in ["primary_score", "acc_norm", "acc_raw", "exact_match", "exact_match_flex"]:
        val = row.get(key)
        if val is not None and pd.notna(val):
            return key, float(val)
    return None, None


def _resolve_output_root() -> Path:
    root = globals().get("FT_HS_OLMES_OUT")
    if root is not None and (Path(root) / "combined_summary.json").exists():
        return Path(root)

    base = Path(globals().get("OUTPUT_BASE", Path.cwd() / "experiments" / "olmes"))
    candidates = sorted(base.glob("olmes_finetuned_hs_gsm8k_hellaswag_*"), key=lambda p: p.stat().st_mtime, reverse=True)
    for c in candidates:
        if (c / "combined_summary.json").exists():
            globals()["FT_HS_OLMES_OUT"] = c
            return c
    raise FileNotFoundError("Kein combined_summary.json gefunden. Fuehre zuerst die OLMES-Eval-Zelle aus.")


def _resolve_task_output(root: Path, task_key: str, fallback_subdir: str) -> Path:
    summary_path = root / "combined_summary.json"
    if summary_path.exists():
        summary = _read_json(summary_path)
        task_outputs = summary.get("task_outputs", {})
        p = task_outputs.get(task_key)
        if p and (Path(p) / "metrics.json").exists():
            return Path(p)

    fallback = root / fallback_subdir
    if (fallback / "metrics.json").exists():
        return fallback
    raise FileNotFoundError(f"metrics.json fuer {task_key} nicht gefunden unter {fallback}")


def _task_row_from_metrics(metrics_path: Path, expected_task: str):
    metrics = _read_json(metrics_path)
    entries = _extract_task_entries(metrics)

    if not entries:
        return {
            "task": expected_task,
            "task_norm": _normalize_task_name(expected_task),
            "accuracy_metric": None,
            "accuracy": None,
            "source": str(metrics_path),
        }

    chosen = None
    for e in entries:
        name = e.get("alias") or e.get("task_name") or e.get("name") or e.get("id") or "unknown_task"
        if expected_task.split("::")[0] in _normalize_task_name(name):
            chosen = e
            break
    if chosen is None:
        chosen = entries[0]

    m = chosen.get("metrics", {}) or {}
    task_name = chosen.get("alias") or chosen.get("task_name") or chosen.get("name") or chosen.get("id") or expected_task
    row = {
        "task": task_name,
        "task_norm": _normalize_task_name(task_name),
        "primary_score": m.get("primary_score", chosen.get("primary_score")),
        "acc_raw": m.get("acc_raw", chosen.get("acc_raw")),
        "acc_norm": m.get("acc_norm", chosen.get("acc_norm")),
        "exact_match": m.get("exact_match", chosen.get("exact_match")),
        "exact_match_flex": m.get("exact_match_flex", chosen.get("exact_match_flex")),
        "num_instances": chosen.get("num_instances"),
        "processing_time_sec": chosen.get("processing_time"),
        "source": str(metrics_path),
    }
    metric_name, metric_val = _pick_accuracy_metric(task_name, row)
    row["accuracy_metric"] = metric_name
    row["accuracy"] = metric_val
    return row


root = _resolve_output_root()
gsm_metrics = _resolve_task_output(root, "gsm8k::olmes", "gsm8k") / "metrics.json"
hsw_metrics = _resolve_task_output(root, "hellaswag::olmo1", "hellaswag") / "metrics.json"

rows = [
    _task_row_from_metrics(gsm_metrics, "gsm8k::olmes"),
    _task_row_from_metrics(hsw_metrics, "hellaswag::olmo1"),
]
ft_df = pd.DataFrame(rows)
display(ft_df)

plot_df = ft_df.dropna(subset=["accuracy"]).copy()
if plot_df.empty:
    print("Keine plotbaren Accuracy-Werte gefunden.")
else:
    plt.figure(figsize=(9, 4))
    plt.bar(plot_df["task"], plot_df["accuracy"])
    plt.title("Fine-tuned HellaSwag Model (OLMES): GSM8K vs HellaSwag")
    plt.ylabel("accuracy")
    plt.xlabel("task")
    plt.ylim(0, 1)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

gsm = ft_df[ft_df["task_norm"].str.contains("gsm8k", na=False)]
hsw = ft_df[ft_df["task_norm"].str.contains("hellaswag", na=False)]

if not gsm.empty and not hsw.empty and pd.notna(gsm.iloc[0]["accuracy"]) and pd.notna(hsw.iloc[0]["accuracy"]):
    gsm_row = gsm.iloc[0]
    hsw_row = hsw.iloc[0]
    print("\nAccuracy-Vergleich (Fine-tuned HellaSwag Modell):")
    print(f"- GSM8K ({gsm_row['accuracy_metric']}): {gsm_row['accuracy']:.4f}")
    print(f"- HellaSwag ({hsw_row['accuracy_metric']}): {hsw_row['accuracy']:.4f}")
    print(f"- Differenz (HellaSwag - GSM8K): {hsw_row['accuracy'] - gsm_row['accuracy']:+.4f}")
else:
    print("Vergleich unvollstaendig: Mindestens ein Task ohne valide Accuracy.")
    print("GSM8K metrics:", gsm_metrics)
    print("HellaSwag metrics:", hsw_metrics)

print("Fine-tuned HellaSwag OLMES output root:", root)
print("- GSM8K metrics:", gsm_metrics)
print("- HellaSwag metrics:", hsw_metrics)

### Option D++: Sequential Fine-Tuning (2 Phasen) – Attention dann MLP

Zuerst Attention [18-21] optimieren, dann auf diesem stabilen State die MLP [21] optimieren.
Das verhindert Catastrophic Forgetting und balanciert Context-Integration mit Wissensupdate.

| Phase | Komponente | Layers | Params | Generationen | σ | Laufzeit |
|---|---|---|---|---|---|---|
| **1** | Attention | [18-21] | ~7.2M | 60 | 0.003→0.001 | ~50 min (A100) |
| **2** | MLP | [21] | ~12M | 60 | 0.001→0.0005 | ~60 min (A100) |
| **Total** | Combined | [18-21] Attn + [21] MLP | ~19.2M | 120 | – | ~110 min |

**Vorteil:** Phase 1 stärkt Context-Integration (Attention) ohne Störung. Phase 2 dann Faktenwissen (MLP) auf stabilem Grund.
**Risiko:** Phase 2 könnte Phase 1 partiell überschreiben → daher kleineres σ in Phase 2.


In [ ]:
# ── Option D++: Phase 1 – ES-Training für Attention [18-21] ──
# Ziel: Context-Integration verbessern (Attention stärken)
# Budget: ~50 Min auf A100

import torch, time
from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES
from es_llm.fitness.gsm8k_loglikelihood import GSM8KLogLikelihoodFitness

print("\n" + "="*70)
print("  PHASE 1: ES-Training für Attention [18-21] (Context-Integration)")
print("="*70)

# 1) Base-Modell laden (neu, unbefestigt)
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

# 2) Layer auswählen – NUR Attention [18-21] (~7.2M Parameter)
selector_p1 = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[18, 19, 20, 21],
    components="attention",
)
print(f"\nTarget Phase 1: {selector_p1.num_target_elements:,} Parameter (Attention [18-21])")
print(selector_p1.summary())

# 3) ES-Algorithmus für Phase 1
NUM_GEN_P1 = 60

es_p1 = OpenAIES(
    population_size=96,
    sigma=0.003,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
    weight_decay=0.002,
    optimizer="adam",
    sigma_decay="cosine",
    sigma_final=0.001,
    max_generations=NUM_GEN_P1,
)

# 4) Fitness-Funktion für Phase 1
fitness_fn_p1 = GSM8KLogLikelihoodFitness(
    num_samples=100,
    split="train",
    target_mode="full",
    batch_size=8,
    reshuffle=True,
    pool_size=5000,
)

# 5) Training Loop Phase 1
current_params_p1 = selector_p1.get_flat_params().clone()
best_ever_p1 = -float("inf")
best_ever_params_p1 = current_params_p1.clone()
history_p1 = []

print(f"\n{'='*70}")
print(f"  Phase 1 Start: Attention [18-21] | σ={es_p1.sigma:.4f}")
print(f"{'='*70}\n")

t_total_p1 = time.time()

for gen in range(1, NUM_GEN_P1 + 1):
    t0 = time.time()
    fitness_fn_p1.reshuffle_data()

    candidates = es_p1.ask(current_params_p1)
    fitnesses = []
    for cand in candidates:
        selector_p1.set_flat_params(cand)
        fit = fitness_fn_p1.evaluate(model, tokenizer)
        fitnesses.append(fit)

    result = es_p1.tell(current_params_p1, candidates, fitnesses)
    current_params_p1 = result.new_params.clone()
    selector_p1.set_flat_params(current_params_p1)

    if result.best_fitness > best_ever_p1:
        best_ever_p1 = result.best_fitness
        best_ever_params_p1 = current_params_p1.clone()

    elapsed = time.time() - t0
    history_p1.append({
        "gen": gen, "best": result.best_fitness, "mean": result.mean_fitness,
        "std": result.std_fitness, "best_ever": best_ever_p1,
        "sigma": es_p1.current_sigma, "elapsed": elapsed,
    })

    if gen % 10 == 0 or gen == 1:
        total_t = time.time() - t_total_p1
        eta = (total_t / gen) * (NUM_GEN_P1 - gen)
        print(f"P1 Gen {gen:3d}/{NUM_GEN_P1} | best={result.best_fitness:.4f}  "
              f"mean={result.mean_fitness:.4f}  best_ever={best_ever_p1:.4f}  "
              f"σ={es_p1.current_sigma:.5f}  [{elapsed:.1f}s/gen, ETA ~{eta/60:.0f}min]")

    torch.cuda.empty_cache()

# Phase 1 abschließen
selector_p1.set_flat_params(best_ever_params_p1)
total_time_p1 = time.time() - t_total_p1
print(f"\n✅ PHASE 1 abgeschlossen ({total_time_p1/60:.1f} min)")
print(f"   Best-ever LL: {best_ever_p1:.4f}")

# ── SPEICHERN: Model nach Phase 1 ──
model_state_p1 = {name: p.data.clone() for name, p in model.named_parameters()}
torch.save(model_state_p1, "/content/model_after_phase1.pt")
print(f"\n📦 Model nach Phase 1 gespeichert: /content/model_after_phase1.pt")

# ──────────────────────────────────────────────────────────────────
# PHASE 2: MLP [21] auf gespeichertem Phase-1-State
# ──────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("  PHASE 2: ES-Training für MLP [21] (Faktenwissen)")
print("  → Startet von Phase-1-State (Attention bereits optimiert)")
print("="*70)

# 1) LADE Phase-1-Model
phase1_state = torch.load("/content/model_after_phase1.pt", weights_only=False)
for name, param in model.named_parameters():
    if name in phase1_state:
        param.data.copy_(phase1_state[name])
print(f"\n✅ Phase-1-State geladen ins Modell")

# 2) Layer auswählen – NUR MLP [21] (~12M Parameter)
selector_p2 = LayerSelector(
    model=model,
    strategy="by_index",
    layer_indices=[21],
    components="mlp",
)
print(f"\nTarget Phase 2: {selector_p2.num_target_elements:,} Parameter (MLP [21])")
print(selector_p2.summary())

# 3) ES-Algorithmus für Phase 2 (konservativer, kleineres σ)
NUM_GEN_P2 = 60

es_p2 = OpenAIES(
    population_size=96,
    sigma=0.001,                    # Kleiner als Phase 1: vorsichtig auf Phase-1-State
    learning_rate=0.0005,           # Auch konservativer
    antithetic=True,
    fitness_shaping="centered_rank",
    weight_decay=0.001,
    optimizer="adam",
    sigma_decay="cosine",
    sigma_final=0.0005,
    max_generations=NUM_GEN_P2,
)

# 4) Fitness-Funktion für Phase 2 (neuer Pool-Shuffle)
fitness_fn_p2 = GSM8KLogLikelihoodFitness(
    num_samples=120,
    split="train",
    target_mode="full",
    batch_size=12,
    reshuffle=True,
    pool_size=5000,
)

# 5) Training Loop Phase 2
current_params_p2 = selector_p2.get_flat_params().clone()
best_ever_p2 = -float("inf")
best_ever_params_p2 = current_params_p2.clone()
history_p2 = []

print(f"\n{'='*70}")
print(f"  Phase 2 Start: MLP [21] | σ={es_p2.sigma:.6f}")
print(f"  (Aufbauend auf Phase 1 – Attention [18-21] bleibt gefestigt)")
print(f"{'='*70}\n")

t_total_p2 = time.time()

for gen in range(1, NUM_GEN_P2 + 1):
    t0 = time.time()
    fitness_fn_p2.reshuffle_data()

    candidates = es_p2.ask(current_params_p2)
    fitnesses = []
    for cand in candidates:
        selector_p2.set_flat_params(cand)
        fit = fitness_fn_p2.evaluate(model, tokenizer)
        fitnesses.append(fit)

    result = es_p2.tell(current_params_p2, candidates, fitnesses)
    current_params_p2 = result.new_params.clone()
    selector_p2.set_flat_params(current_params_p2)

    if result.best_fitness > best_ever_p2:
        best_ever_p2 = result.best_fitness
        best_ever_params_p2 = current_params_p2.clone()

    elapsed = time.time() - t0
    history_p2.append({
        "gen": gen, "best": result.best_fitness, "mean": result.mean_fitness,
        "std": result.std_fitness, "best_ever": best_ever_p2,
        "sigma": es_p2.current_sigma, "elapsed": elapsed,
    })

    if gen % 10 == 0 or gen == 1:
        total_t = time.time() - t_total_p2
        eta = (total_t / gen) * (NUM_GEN_P2 - gen)
        print(f"P2 Gen {gen:3d}/{NUM_GEN_P2} | best={result.best_fitness:.4f}  "
              f"mean={result.mean_fitness:.4f}  best_ever={best_ever_p2:.4f}  "
              f"σ={es_p2.current_sigma:.6f}  [{elapsed:.1f}s/gen, ETA ~{eta/60:.0f}min]")

    torch.cuda.empty_cache()

# Phase 2 abschließen
selector_p2.set_flat_params(best_ever_params_p2)
total_time_p2 = time.time() - t_total_p2
print(f"\n✅ PHASE 2 abgeschlossen ({total_time_p2/60:.1f} min)")
print(f"   Best-ever LL: {best_ever_p2:.4f}")

# ── SPEICHERN: Final Model nach Phase 2 ──
model_state_final = {name: p.data.clone() for name, p in model.named_parameters()}
torch.save(model_state_final, "/content/model_final.pt")
print(f"\n📦 Final Model gespeichert: /content/model_final.pt")

print(f"\n{'='*70}")
print(f"  TRAINING COMPLETE: Sequential 2-Phase")
print(f"  Phase 1: {total_time_p1/60:.1f} min (Attention [18-21])")
print(f"  Phase 2: {total_time_p2/60:.1f} min (MLP [21])")
print(f"  Total:   {(total_time_p1 + total_time_p2)/60:.1f} min")
print(f"{'='*70}")


In [ ]:
# ── Evaluation: Final Model (nach Phase 1 + Phase 2) vs Base-Modell ──

from es_llm.fitness.gsm8k import GSM8KFitness
from es_llm.model.loader import load_model_and_tokenizer as load_base
import torch, time

NUM_EVAL = 200

print("\n" + "="*70)
print("  EVALUATION: Final Model vs Base-Modell")
print("="*70)

# --- 1) Final Model laden (aus Phase 2) ---
print(f"\n1. Final Model laden (Phase 2 State)...")
final_state = torch.load("/content/model_final.pt", weights_only=False)
for name, param in model.named_parameters():
    if name in final_state:
        param.data.copy_(final_state[name])
print(f"   ✅ Final Model geladen")

# --- 2) Final Model evaluieren ---
print(f"\n2. Evaluiere FINAL MODEL auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
final_acc = eval_fn.evaluate(model, tokenizer)
t_final = time.time() - t0
print(f"   Final Accuracy: {final_acc:.2%}  ({t_final:.0f}s)")

# --- 3) Base-Modell laden und evaluieren ---
print(f"\n3. Lade Base-Modell für Vergleich...")
base_model, base_tokenizer = load_base(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"\n4. Evaluiere BASE-MODELL auf {NUM_EVAL} GSM8K-TEST-Fragen...")
eval_fn_base = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
base_acc = eval_fn_base.evaluate(base_model, base_tokenizer)
t_base = time.time() - t0
print(f"   Base Accuracy:  {base_acc:.2%}  ({t_base:.0f}s)")

# Base-Modell freigeben
del base_model, base_tokenizer
torch.cuda.empty_cache()

# --- 4) Ergebnis ---
diff = final_acc - base_acc
print(f"\n{'='*70}")
print(f"  ERGEBNISSE AUF GSM8K TEST-SPLIT (n={NUM_EVAL})")
print(f"  Training: TRAIN-Split | Evaluation: TEST-Split (kein Data Leakage)")
print(f"{'='*70}")
print(f"  Base-Modell:           {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Final (Phase 1+2):     {final_acc:6.2%}  ({int(final_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Verbesserung:          {diff:+6.2%}")
print(f"{'='*70}")


In [ ]:
# ── Visualisierung: Option D++ (Phase 1 + Phase 2) + Accuracy ──

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)

# ── Plot 1: Phase 1 Fitness ──
ax1 = fig.add_subplot(gs[0, 0])
gens_p1 = [h["gen"] for h in history_p1]
bests_p1 = [h["best"] for h in history_p1]
means_p1 = [h["mean"] for h in history_p1]
best_evers_p1 = [h["best_ever"] for h in history_p1]

ax1.plot(gens_p1, bests_p1, label="Best Fitness", linewidth=1.5, alpha=0.7, color="#2196F3")
ax1.plot(gens_p1, means_p1, label="Mean Fitness", linewidth=1.5, alpha=0.5, color="#FF9800")
ax1.plot(gens_p1, best_evers_p1, label="Best-Ever", linewidth=2, color="#4CAF50", linestyle="--")
ax1.set_xlabel("Generation")
ax1.set_ylabel("Log-Likelihood")
ax1.set_title("Phase 1: Attention [18-21]\n(Context-Integration)")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Phase 2 Fitness ──
ax2 = fig.add_subplot(gs[0, 1])
gens_p2 = [h["gen"] for h in history_p2]
bests_p2 = [h["best"] for h in history_p2]
means_p2 = [h["mean"] for h in history_p2]
best_evers_p2 = [h["best_ever"] for h in history_p2]

ax2.plot(gens_p2, bests_p2, label="Best Fitness", linewidth=1.5, alpha=0.7, color="#FF6F00")
ax2.plot(gens_p2, means_p2, label="Mean Fitness", linewidth=1.5, alpha=0.5, color="#FFA726")
ax2.plot(gens_p2, best_evers_p2, label="Best-Ever", linewidth=2, color="#D32F2F", linestyle="--")
ax2.set_xlabel("Generation")
ax2.set_ylabel("Log-Likelihood")
ax2.set_title("Phase 2: MLP [21]\n(Faktenwissen)")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── Plot 3: Combined Sigma-Schedule ──
ax3 = fig.add_subplot(gs[0, 2])
sigmas_p1 = [h["sigma"] for h in history_p1]
sigmas_p2 = [h["sigma"] for h in history_p2]

# Phase 1: gens_p1 normal, Phase 2: gens_p2 + len(gens_p1)
gens_combined = list(gens_p1) + [g + len(gens_p1) for g in gens_p2]
sigmas_combined = sigmas_p1 + sigmas_p2

ax3.plot(gens_p1, sigmas_p1, linewidth=2, color="#2196F3", label="Phase 1: σ 0.003→0.001")
ax3.plot([g + len(gens_p1) for g in gens_p2], sigmas_p2, linewidth=2, color="#D32F2F", label="Phase 2: σ 0.001→0.0005")
ax3.axvline(x=len(gens_p1), color="gray", linestyle=":", linewidth=2, alpha=0.5, label="Phase-Grenze")
ax3.set_xlabel("Generation (kombiniert)")
ax3.set_ylabel("σ (Noise Std)")
ax3.set_title("Cosine σ-Decay Schedule\n(beide Phasen)")
ax3.grid(True, alpha=0.3)
ax3.legend(fontsize=8)

# ── Plot 4: Accuracy-Vergleich ──
ax4 = fig.add_subplot(gs[1, 0:2])
labels = ["Base-Modell\n(unverändert)", "Final Model\n(Phase 1 + Phase 2)"]
accs = [base_acc, final_acc]
colors = ["#4C72B0", "#55A868"]

bars = ax4.bar(labels, accs, color=colors, width=0.5, edgecolor="black", linewidth=1, alpha=0.85)
for bar, acc in zip(bars, accs):
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f"{acc:.1%}\n({int(acc*NUM_EVAL)}/{NUM_EVAL})",
             ha="center", va="bottom", fontweight="bold", fontsize=12)

ax4.set_ylabel("Accuracy", fontsize=11)
ax4.set_title(f"GSM8K TEST Accuracy (n={NUM_EVAL})\nTrain: train-Split | Eval: test-Split", fontsize=12)
ax4.set_ylim(0, max(accs) * 1.3)
ax4.axhline(y=base_acc, color="red", linestyle="--", alpha=0.3, linewidth=1)
ax4.grid(axis="y", alpha=0.3)

# ── Plot 5: Laufzeit Summary ──
ax5 = fig.add_subplot(gs[1, 2])
times_p1 = [h["elapsed"] for h in history_p1]
times_p2 = [h["elapsed"] for h in history_p2]

phases = ["Phase 1\n(Attention\n[18-21])", "Phase 2\n(MLP\n[21])"]
total_times = [sum(times_p1), sum(times_p2)]
colors_phases = ["#2196F3", "#D32F2F"]

bars_time = ax5.bar(phases, total_times, color=colors_phases, width=0.6, edgecolor="black", linewidth=1)
for bar, t in zip(bars_time, total_times):
    ax5.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
             f"{t/60:.1f} min\n({int(t/len(times_p1))}s/gen avg)",
             ha="center", va="bottom", fontweight="bold", fontsize=10)

ax5.set_ylabel("Total Time (seconds)", fontsize=10)
ax5.set_title("Training Duration\n(komplett)", fontsize=11)
ax5.axhline(y=sum(total_times)/2, color="gray", linestyle="--", alpha=0.3)
ax5.grid(axis="y", alpha=0.3)

fig.suptitle("Option D++: Sequential Fine-Tuning (2 Phasen)\nPhase 1: Attention [18-21] → Phase 2: MLP [21]",
             fontsize=15, fontweight="bold", y=0.995)
plt.tight_layout()
plt.show()

# ── Zusammenfassung ──
print(f"\n{'='*70}")
print(f"  ZUSAMMENFASSUNG – Option D++ (Sequential 2-Phase)")
print(f"{'='*70}")
print(f"\n  PHASE 1: Attention [18-21]")
print(f"  ──────────────────────────")
print(f"  Generationen:      {len(history_p1)}")
print(f"  Population:        96")
print(f"  Parameter:         7,200,000 (Attention only)")
print(f"  σ-Schedule:        0.003 → 0.001 (cosine)")
print(f"  Best-Ever LL:      {best_ever_p1:.4f}")
print(f"  Total Zeit:        {sum(times_p1):.0f}s ({sum(times_p1)/60:.1f} min)")
print(f"  Ø Zeit/Gen:        {np.mean(times_p1):.1f}s")

print(f"\n  PHASE 2: MLP [21] (auf Phase-1-State)")
print(f"  ────────────────────────────────────")
print(f"  Generationen:      {len(history_p2)}")
print(f"  Population:        96")
print(f"  Parameter:         12,000,000 (MLP only [21])")
print(f"  σ-Schedule:        0.001 → 0.0005 (cosine, konservativer)")
print(f"  Best-Ever LL:      {best_ever_p2:.4f}")
print(f"  Total Zeit:        {sum(times_p2):.0f}s ({sum(times_p2)/60:.1f} min)")
print(f"  Ø Zeit/Gen:        {np.mean(times_p2):.1f}s")

total_combined = sum(times_p1) + sum(times_p2)
print(f"\n  GESAMT (Phase 1 + Phase 2)")
print(f"  ──────────────────────────")
print(f"  Total Generationen: {len(history_p1) + len(history_p2)}")
print(f"  Total Parameter:    19,200,000 (7.2M Attn + 12M MLP)")
print(f"  Total Zeit:         {total_combined:.0f}s ({total_combined/60:.1f} min)")

print(f"\n  ACCURACY RESULTS (TEST-Split, n={NUM_EVAL})")
print(f"  ──────────────────────────────────────────")
print(f"  Base-Modell:       {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Final Model:       {final_acc:6.2%}  ({int(final_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Verbesserung:      {diff:+6.2%}  ({diff*NUM_EVAL:+.0f} Fragen)")
print(f"{'='*70}")


In [ ]:
# ── Visualisierung: v5 Training + Accuracy-Vergleich ──

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

# ── Plot 1: Fitness über Generationen ──
ax1 = fig.add_subplot(gs[0, 0])
gens = [h["gen"] for h in history]
bests = [h["best"] for h in history]
means = [h["mean"] for h in history]
best_evers = [h["best_ever"] for h in history]

ax1.plot(gens, bests, label="Best Fitness", linewidth=1.5, alpha=0.7, color="#2196F3")
ax1.plot(gens, means, label="Mean Fitness", linewidth=1.5, alpha=0.5, color="#FF9800")
ax1.plot(gens, best_evers, label="Best-Ever", linewidth=2, color="#4CAF50", linestyle="--")
ax1.set_xlabel("Generation")
ax1.set_ylabel("Log-Likelihood Fitness")
ax1.set_title("ES Training v5 – Fitness über Generationen\n(4 Layers [18-21] + Cosine σ)")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Sigma-Schedule ──
ax2 = fig.add_subplot(gs[0, 1])
sigmas = [h["sigma"] for h in history]
ax2.plot(gens, sigmas, linewidth=2, color="#9C27B0")
ax2.set_xlabel("Generation")
ax2.set_ylabel("σ (Noise Std)")
ax2.set_title("Cosine σ-Decay: 0.003 → 0.001")
ax2.grid(True, alpha=0.3)
ax2.fill_between(gens, sigmas, alpha=0.15, color="#9C27B0")

# ── Plot 3: Accuracy-Vergleich ──
ax3 = fig.add_subplot(gs[1, 0])
labels = ["Base-Modell\n(unverändert)", "Fine-tuned v5\n(4 Layers, 100 Gen)"]
accs = [base_acc, finetuned_acc]
colors = ["#4C72B0", "#55A868"]

bars = ax3.bar(labels, accs, color=colors, width=0.5, edgecolor="black", linewidth=0.8)
for bar, acc in zip(bars, accs):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{acc:.1%}\n({int(acc*NUM_EVAL)}/{NUM_EVAL})",
             ha="center", va="bottom", fontweight="bold", fontsize=11)

ax3.set_ylabel("Accuracy")
ax3.set_title(f"GSM8K TEST Accuracy (n={NUM_EVAL})")
ax3.set_ylim(0, max(accs) * 1.3)
ax3.axhline(y=base_acc, color="red", linestyle="--", alpha=0.3, label="Baseline")
ax3.grid(axis="y", alpha=0.3)
ax3.legend(fontsize=9)

# ── Plot 4: Zeitverlauf pro Generation ──
ax4 = fig.add_subplot(gs[1, 1])
times = [h["elapsed"] for h in history]
ax4.plot(gens, times, linewidth=1.5, color="#FF5722", alpha=0.7)
ax4.axhline(y=np.mean(times), color="gray", linestyle="--", alpha=0.5,
            label=f"Ø {np.mean(times):.1f}s/gen")
ax4.set_xlabel("Generation")
ax4.set_ylabel("Zeit (Sekunden)")
ax4.set_title("Laufzeit pro Generation")
ax4.grid(True, alpha=0.3)
ax4.legend(fontsize=9)

fig.suptitle("ES-Training v5 – 4 Layers [18-21] + Cosine σ + Full Reasoning",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Zusammenfassung ──
total_time = sum(times)
print(f"\n{'='*65}")
print(f"  ZUSAMMENFASSUNG – ES-Training v5")
print(f"{'='*65}")
print(f"  Generationen:      {len(history)}")
print(f"  Population:        {es.population_size}")
print(f"  Parameter:         {selector.num_target_elements:,} (Layer 18-21 Attention)")
print(f"  Gesamtzeit:        {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"  Ø pro Generation:  {np.mean(times):.1f}s")
print(f"  σ Start → Ende:   {history[0]['sigma']:.5f} → {history[-1]['sigma']:.5f}")
print(f"  Best-Ever LL:      {history[-1]['best_ever']:.4f}")
print(f"  Data Reshuffling:  ON (120 neue Fragen/Gen aus Pool von 5000)")
print(f"  Batch-Size:        12")

In [ ]:
# ── Vergleich: Fine-tuned vs Base-Modell auf GSM8K (Accuracy) ──
# Testet das gerade trainierte Modell gegen das unveränderte Base-Modell
# mit jeweils 100 Fragen aus dem TEST-Split.

from es_llm.fitness.gsm8k import GSM8KFitness
import torch, time
import matplotlib.pyplot as plt

NUM_EVAL = 100

# --- 1) Fine-tuned Modell evaluieren (aktuell im Speicher) ---
print(f"Evaluiere FINE-TUNED Modell auf {NUM_EVAL} GSM8K-Testfragen...")
eval_fn = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
finetuned_acc = eval_fn.evaluate(model, tokenizer)
t_ft = time.time() - t0
print(f"  Fine-tuned Accuracy: {finetuned_acc:.2%}  ({t_ft:.0f}s)")

# --- 2) Base-Modell laden und evaluieren ---
print(f"\nLade Base-Modell fuer Vergleich...")
from es_llm.model.loader import load_model_and_tokenizer as load_base
base_model, base_tokenizer = load_base(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype="float16", device="cuda"
)

print(f"Evaluiere BASE-Modell auf {NUM_EVAL} GSM8K-Testfragen...")
eval_fn_base = GSM8KFitness(num_samples=NUM_EVAL, split="test", max_new_tokens=256)

t0 = time.time()
base_acc = eval_fn_base.evaluate(base_model, base_tokenizer)
t_base = time.time() - t0
print(f"  Base Accuracy:       {base_acc:.2%}  ({t_base:.0f}s)")

# Base-Modell freigeben
del base_model, base_tokenizer
torch.cuda.empty_cache()

# --- 3) Ergebnis ---
diff = finetuned_acc - base_acc
print(f"\n{'='*50}")
print(f"  Base-Modell:     {base_acc:6.2%}  ({int(base_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Fine-tuned:      {finetuned_acc:6.2%}  ({int(finetuned_acc*NUM_EVAL)}/{NUM_EVAL})")
print(f"  Differenz:       {diff:+6.2%}")
print(f"{'='*50}")

# --- 4) Plot ---
labels = ["Base-Modell", "Fine-tuned\n(ES + LL)"]
accs = [base_acc, finetuned_acc]
colors = ["#4C72B0", "#55A868"]

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(labels, accs, color=colors, width=0.5, edgecolor="black", linewidth=0.8)

# Werte auf die Balken schreiben
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{acc:.1%}\n({int(acc*NUM_EVAL)}/{NUM_EVAL})",
            ha="center", va="bottom", fontweight="bold", fontsize=12)

ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title(f"GSM8K Accuracy – Base vs Fine-tuned (n={NUM_EVAL})", fontsize=13)
ax.set_ylim(0, max(accs) * 1.25)
ax.axhline(y=base_acc, color="red", linestyle="--", alpha=0.4, label="Baseline")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Ergebnisse visualisieren

In [ ]:
import matplotlib.pyplot as plt
import json

# Falls history nicht im Speicher ist (z.B. nach Kernel-Neustart):
# → Aus training_log.jsonl laden (Option A) oder manuell definieren
if "history" not in dir():
    # Option A: Aus JSONL-Datei laden
    from pathlib import Path
    log_path = Path("/content/experiments/runs")
    # Neuesten Run finden
    if log_path.exists():
        runs = sorted(log_path.iterdir(), key=lambda p: p.name, reverse=True)
        for run in runs:
            jsonl = run / "training_log.jsonl"
            if jsonl.exists():
                with open(jsonl) as f:
                    history = [json.loads(l) for l in f]
                print(f"History geladen aus: {jsonl}")
                break
    if "history" not in dir():
        raise RuntimeError(
            "Variable 'history' nicht gefunden!\n"
            "Bitte zuerst eine Trainings-Zelle ausfuehren (Option A, B1 oder B2)."
        )

gens = [h.get("gen") or h.get("generation") for h in history]
bests = [h.get("best") or h.get("best_fitness") for h in history]
means = [h.get("mean") or h.get("mean_fitness") for h in history]

plt.figure(figsize=(10, 5))
plt.plot(gens, bests, label="Best Fitness", linewidth=2)
plt.plot(gens, means, label="Mean Fitness", alpha=0.7)
if "baseline_acc" in dir():
    plt.axhline(y=baseline_acc, color="red", linestyle="--", label="Baseline")
plt.xlabel("Generation")
plt.ylabel("Fitness")
plt.title("ES Training – Fitness über Generationen")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Fine-tuned Modell evaluieren

In [ ]:
# Evaluation auf dem Test-Split (größeres Subset)
eval_fitness = GSM8KFitness(
    num_samples=100,
    split="test",
    max_new_tokens=256,
)

final_acc = eval_fitness.evaluate(model, tokenizer)
print(f"\n📊 Ergebnisse:")
print(f"   Baseline (test, n=30):   {baseline_acc:.2%}")
print(f"   Fine-tuned (test, n=100): {final_acc:.2%}")
print(f"   Differenz:                {final_acc - baseline_acc:+.2%}")

## 8. Modell & Ergebnisse speichern

In [ ]:
# ── Checkpoint speichern ──
import json
from pathlib import Path

save_dir = Path("/content/experiments/final")
save_dir.mkdir(parents=True, exist_ok=True)

# Layer-Gewichte speichern
layer_state = {name: p.data.cpu().clone() for name, p in selector.get_target_params()}
torch.save(layer_state, save_dir / "best_layer_weights.pt")

# Komplettes Modell speichern
model.save_pretrained(save_dir / "model")
tokenizer.save_pretrained(save_dir / "model")

# Ergebnisse speichern
results = {
    "baseline_accuracy": baseline_acc,
    "final_accuracy": final_acc,
    "target_layers": selector.target_names,
    "num_target_params": selector.num_target_elements,
    "history": history,
}
(save_dir / "results.json").write_text(json.dumps(results, indent=2, default=str))

print(f"✅ Gespeichert in: {save_dir}")
!ls -la {save_dir}

In [ ]:
# ── Optional: Nach Google Drive kopieren ──
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/es-llm-experiments"
!mkdir -p {DRIVE_DIR}
!cp -r {save_dir}/* {DRIVE_DIR}/
print(f"✅ Kopiert nach Google Drive: {DRIVE_DIR}")

---

## Tipps für Colab

| Thema | Empfehlung |
|-------|------------|
| **GPU-Typ** | T4 (free) reicht für Qwen 0.5B. A100 (Pro) für schnellere Inferenz |
| **dtype** | Immer `float16` auf GPU → halber Speicher, schneller |
| **VRAM** | Qwen 0.5B in fp16 ≈ 1 GB. T4 hat 16 GB → viel Headroom |
| **Timeout** | Colab trennt nach ~90min Inaktivität. Halte Tab offen |
| **Ergebnisse** | Immer nach Google Drive speichern – Colab ist flüchtig! |

### Konfiguration: Voller Layer (~14.9M Parameter)

| Einstellung | Quick-Test | Experiment |
|-------------|-----------|------------|
| **components** | `"all"` | `"all"` |
| **layer_indices** | [23] | [23] |
| **σ** | 0.001 | 0.001 |
| **learning_rate** | 0.001 | 0.001 |
| **population_size** | 10 | 20 |
| **num_samples** | 15 | 25 |
| **num_generations** | 10 | 50 |
| **Inferenzen gesamt** | ~1.500 | ~25.000 |

**Wichtig bei 14.9M Dimensionen:** σ muss **sehr klein** sein (0.001), weil eine Perturbation über 14.9M Parameter sonst das Modellverhalten zerstört.

## 10. Multi-Task Fine-Tuning: Minerva Algebra + HellaSwag

Dieser Block trainiert `Qwen/Qwen2.5-0.5B-Instruct` auf einer kombinierten Trainingsmenge aus Algebra- und HellaSwag-Beispielen und evaluiert danach mit OLMES auf:
- `minerva_math_algebra`
- `hellaswag`

Hinweis:
- Das ist ein pragmatisches SFT-Setup (LoRA) fuer schnelle Experimente.
- OLMES muss in der Runtime verfuegbar sein.

In [ ]:
# Abhaengigkeiten fuer Multi-Task-Finetuning (hard reset)
# Fix fuer gemischte transformers-Installationen in Colab

import site
import glob
import shutil
from pathlib import Path

# 1) Alte Paketordner physisch entfernen (wichtig bei inkonsistenter Installation)
site_paths = site.getsitepackages() + [site.getusersitepackages()]
patterns = [
    "transformers*",
    "tokenizers*",
    "accelerate*",
    "peft*",
    "datasets*",
    "huggingface_hub*",
]

for sp in site_paths:
    for pat in patterns:
        for p in glob.glob(str(Path(sp) / pat)):
            try:
                if Path(p).is_dir():
                    shutil.rmtree(p, ignore_errors=True)
                else:
                    Path(p).unlink(missing_ok=True)
            except Exception:
                pass

# 2) Deinstallieren + sauber neu installieren
%pip uninstall -y -q transformers tokenizers accelerate peft datasets huggingface_hub deepspeed
%pip install -q --no-cache-dir --upgrade --force-reinstall \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "accelerate==1.1.1" \
    "peft==0.13.2" \
    "datasets==3.1.0" \
    "huggingface_hub==0.26.2"

In [ ]:
# Versionen pruefen (ohne direkten Modulimport) + Trainer-Smoketest
import importlib.metadata as im

for pkg in ["transformers", "accelerate", "peft", "datasets", "tokenizers", "huggingface_hub"]:
    try:
        print(f"{pkg}: {im.version(pkg)}")
    except Exception as e:
        print(f"{pkg}: NOT FOUND ({e})")

print("\nWenn du gerade Cell 49 ausgefuehrt hast: Runtime jetzt NEU STARTEN und dann erst weiter.")

# Diesen Block erst NACH Neustart ausfuehren:
# from transformers import Trainer, TrainingArguments
# print("Trainer import OK")

In [ ]:
import os
import json
import random
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM

# PEFT optional halten: wenn Import wegen transformers/peft-Konflikt fehlschlaegt,
# trainieren wir mit teilweisem Unfreeze statt LoRA.
try:
    from peft import LoraConfig, get_peft_model
    PEFT_AVAILABLE = True
    PEFT_IMPORT_ERROR = None
except Exception as e:
    PEFT_AVAILABLE = False
    PEFT_IMPORT_ERROR = str(e)
    print("WARN: PEFT nicht verfuegbar, wechsle auf Fallback-Training ohne LoRA.")
    print("PEFT ImportError:", e)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_BASE = Path("/content/experiments/minerva_hellaswag")
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
FT_OUT = OUTPUT_BASE / f"ft_{RUN_TAG}"
EVAL_OUT = OUTPUT_BASE / f"olmes_eval_{RUN_TAG}"

MAX_LEN = 384
N_ALG_TRAIN = 4000
N_HELLA_TRAIN = 4000
N_ALG_VAL = 400
N_HELLA_VAL = 400

print("FT_OUT:", FT_OUT)
print("EVAL_OUT:", EVAL_OUT)
print("CUDA:", torch.cuda.is_available())
print("PEFT_AVAILABLE:", PEFT_AVAILABLE)

In [ ]:
# 1) Datensaetze laden und in gemeinsames SFT-Format bringen
# Algebra: zuerst hendrycks/competition_math versuchen, dann Fallback EleutherAI/hendrycks_math
# HellaSwag: Rowan/hellaswag

try:
    alg_raw = load_dataset("hendrycks/competition_math", "algebra")
    print("Loaded algebra dataset: hendrycks/competition_math")
except Exception:
    alg_raw = load_dataset("EleutherAI/hendrycks_math", "algebra")
    print("Loaded algebra dataset: EleutherAI/hendrycks_math")

hella_raw = load_dataset("Rowan/hellaswag")


def format_algebra(ex):
    prompt = (
        "You are a math tutor. Solve the algebra problem step by step.\n"
        f"Problem: {ex['problem']}\n"
        "Answer:"
    )
    target = ex.get("solution", "")
    text = f"{prompt} {target}"
    return {"text": text, "source_task": "algebra"}


def format_hellaswag(ex):
    label = int(ex["label"])
    ending = ex["endings"][label]
    prompt = (
        "Choose the most plausible continuation for the context and explain briefly.\n"
        f"Context: {ex['ctx']}\n"
        "Best continuation:"
    )
    text = f"{prompt} {ending}"
    return {"text": text, "source_task": "hellaswag"}

alg_train = alg_raw["train"].shuffle(seed=SEED).select(range(min(N_ALG_TRAIN, len(alg_raw["train"]))))
alg_val = alg_raw["test"].shuffle(seed=SEED).select(range(min(N_ALG_VAL, len(alg_raw["test"]))))

hella_train = hella_raw["train"].shuffle(seed=SEED).select(range(min(N_HELLA_TRAIN, len(hella_raw["train"]))))
hella_val = hella_raw["validation"].shuffle(seed=SEED).select(range(min(N_HELLA_VAL, len(hella_raw["validation"]))))

alg_train = alg_train.map(format_algebra, remove_columns=alg_train.column_names)
alg_val = alg_val.map(format_algebra, remove_columns=alg_val.column_names)
hella_train = hella_train.map(format_hellaswag, remove_columns=hella_train.column_names)
hella_val = hella_val.map(format_hellaswag, remove_columns=hella_val.column_names)

train_ds = concatenate_datasets([alg_train, hella_train]).shuffle(seed=SEED)
val_ds = concatenate_datasets([alg_val, hella_val]).shuffle(seed=SEED)

print("Train size:", len(train_ds), "| Val size:", len(val_ds))
print("Sample:\n", train_ds[0]["text"][:500])

In [ ]:
# 2) Tokenisierung + Modell aufsetzen (LoRA falls verfuegbar, sonst Fallback ohne PEFT)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

if PEFT_AVAILABLE:
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()
    training_mode = "lora"
else:
    model = base_model
    training_mode = "fallback_last_layers"

    # Nur letzte 2 Decoder-Layer + lm_head trainierbar machen (ressourcenschonender).
    for _, p in model.named_parameters():
        p.requires_grad = False

    for name, p in model.named_parameters():
        if "model.layers.22" in name or "model.layers.23" in name or "lm_head" in name:
            p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Fallback aktiv: trainable {trainable:,}/{total:,} ({100*trainable/total:.2f}%)")

def tokenize_batch(batch):
    tok = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )
    tok["labels"] = tok["input_ids"].copy()
    return tok

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_batch, batched=True, remove_columns=val_ds.column_names)

print("Training mode:", training_mode)
print("Tokenized train size:", len(train_tok), "| val size:", len(val_tok))

In [ ]:
# 3) Training starten (manueller Loop, robust fuer LoRA/Fallback)

import math

FT_OUT.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

train_tok = train_tok.with_format("python")
val_tok = val_tok.with_format("python")

def causal_lm_collate(features):
    input_ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
    attention_mask = torch.tensor([f["attention_mask"] for f in features], dtype=torch.long)
    labels = input_ids.clone()
    labels[attention_mask == 0] = -100
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

train_loader = DataLoader(
    train_tok,
    batch_size=2,
    shuffle=True,
    collate_fn=causal_lm_collate,
)
val_loader = DataLoader(
    val_tok,
    batch_size=2,
    shuffle=False,
    collate_fn=causal_lm_collate,
)

# Nur trainierbare Parameter an den Optimizer geben.
trainable_params = [p for p in model.parameters() if p.requires_grad]
if not trainable_params:
    raise RuntimeError("Keine trainierbaren Parameter gefunden.")

lr = 2e-4 if training_mode == "lora" else 5e-5
optimizer = torch.optim.AdamW(trainable_params, lr=lr)
num_epochs = 1
grad_acc_steps = 8

steps_per_epoch = max(1, math.ceil(len(train_loader) / grad_acc_steps))
total_steps = max(1, num_epochs * steps_per_epoch)
warmup_steps = max(1, int(0.03 * total_steps))

def lr_lambda(current_step: int) -> float:
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(0.0, 1.0 - progress)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"Training mode: {training_mode}, lr={lr}")
model.train()
global_step = 0
running_loss = 0.0

for epoch in range(num_epochs):
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss / grad_acc_steps
        loss.backward()
        running_loss += loss.item()

        if step % grad_acc_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % 20 == 0:
                print(f"epoch={epoch+1} step={global_step} train_loss={running_loss/20:.4f}")
                running_loss = 0.0

# einfache Val-Loss Messung
model.eval()
val_losses = []
with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= 100:
            break
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        val_losses.append(float(out.loss.item()))

if val_losses:
    print(f"Validation loss (sampled): {sum(val_losses)/len(val_losses):.4f}")

model.save_pretrained(str(FT_OUT / "model"))
tokenizer.save_pretrained(str(FT_OUT / "model"))

print("Fine-tuned model saved to:", FT_OUT / "model")

### 10.1 ES auf Original-Layern (ohne LoRA) fuer Minerva Algebra + HellaSwag

Dieser Teil trainiert direkt auf den Original-Layern via ES (keine Adapter) und dient als Vergleich gegen LoRA.

Ablauf:
1. Basismodell laden
2. Layer selektieren
3. Multi-Task-Fitness auf Algebra + HellaSwag berechnen
4. ES-Optimierung
5. Modell speichern fuer OLMES-Eval

In [ ]:
import math
import re
from typing import List

from es_llm.model.loader import load_model_and_tokenizer
from es_llm.model.layer_selector import LayerSelector
from es_llm.es.openai_es import OpenAIES

ES_OUT = OUTPUT_BASE / f"es_layer_{RUN_TAG}"
ES_OUT.mkdir(parents=True, exist_ok=True)

# Kleines, stabiles Fitness-Subset fuer ES – reduziert fuer Colab-Effizienz
ALG_ES_N = 20
HELLA_ES_N = 30

alg_es = alg_raw["train"].shuffle(seed=SEED).select(range(min(ALG_ES_N, len(alg_raw["train"]))))
hella_es = hella_raw["train"].shuffle(seed=SEED).select(range(min(HELLA_ES_N, len(hella_raw["train"]))))


def _extract_last_number(text: str):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return nums[-1] if nums else None


def _avg_logprob_for_suffix(model, tokenizer, prompt: str, suffix: str) -> float:
    full = prompt + suffix
    tok_full = tokenizer(full, return_tensors="pt").to(model.device)
    tok_prompt = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model(**tok_full)
        logits = out.logits[:, :-1, :]
        labels = tok_full["input_ids"][:, 1:]
        log_probs = torch.log_softmax(logits, dim=-1)
        tok_lp = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    prompt_len = tok_prompt["input_ids"].shape[1] - 1
    suffix_lp = tok_lp[:, prompt_len:]
    if suffix_lp.numel() == 0:
        return -1e9
    return float(suffix_lp.mean().item())


def es_multitask_fitness(model, tokenizer) -> float:
    model.eval()

    # Algebra: einfache numerische Trefferquote
    alg_correct = 0
    for ex in alg_es:
        prompt = (
            "Solve the algebra problem step by step and output final numeric answer.\n"
            f"Problem: {ex['problem']}\nAnswer:"
        )
        inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=80, do_sample=False)
        out_ids = gen[0, inputs["input_ids"].shape[1]:]
        pred_text = tokenizer.decode(out_ids, skip_special_tokens=True)

        pred = _extract_last_number(pred_text)
        gold = _extract_last_number(ex.get("solution", ""))
        alg_correct += int(pred is not None and gold is not None and pred == gold)

    alg_acc = alg_correct / max(1, len(alg_es))

    # HellaSwag: Kandidat mit hoechster mittlerer Log-Wahrscheinlichkeit
    hella_correct = 0
    for ex in hella_es:
        prompt = f"Context: {ex['ctx']}\nBest continuation: "
        scores = []
        for end in ex["endings"]:
            score = _avg_logprob_for_suffix(model, tokenizer, prompt, end)
            scores.append(score)
        pred_idx = int(max(range(len(scores)), key=lambda i: scores[i]))
        hella_correct += int(pred_idx == int(ex["label"]))

    hella_acc = hella_correct / max(1, len(hella_es))

    # Kombinierte Fitness
    return 0.5 * alg_acc + 0.5 * hella_acc

print("ES fitness subsets:", len(alg_es), "algebra |", len(hella_es), "hellaswag")

In [ ]:
# ES-Training auf Original-Layern

es_model, es_tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    dtype="float16" if torch.cuda.is_available() else "float32",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

# Fokus auf spaete Layer, damit Compute handhabbar bleibt
es_selector = LayerSelector(
    model=es_model,
    strategy="by_index",
    layer_indices=[22, 23],
    components="attention_qkv",
)
print(es_selector.summary())

es_algo = OpenAIES(
    population_size=8,
    sigma=0.001,
    learning_rate=0.001,
    antithetic=True,
    fitness_shaping="centered_rank",
    seed=SEED,
)

ES_GENERATIONS = 3
current_params = es_selector.get_flat_params().clone()
best_params = current_params.clone()

baseline_fit = es_multitask_fitness(es_model, es_tokenizer)
best_fit = baseline_fit
es_history = [{"gen": 0, "best": baseline_fit, "mean": baseline_fit}]
print(f"Baseline multi-task fitness: {baseline_fit:.4f}")

for gen in range(1, ES_GENERATIONS + 1):
    candidates = es_algo.ask(current_params)
    fits = []
    for cand in candidates:
        es_selector.set_flat_params(cand)
        fit = es_multitask_fitness(es_model, es_tokenizer)
        fits.append(fit)

    result = es_algo.tell(current_params, candidates, fits)
    current_params = result.new_params.clone()
    es_selector.set_flat_params(current_params)

    if result.best_fitness > best_fit:
        best_fit = result.best_fitness
        best_params = current_params.clone()

    es_history.append({"gen": gen, "best": result.best_fitness, "mean": result.mean_fitness})
    print(f"ES Gen {gen:02d}/{ES_GENERATIONS} | best={result.best_fitness:.4f} | mean={result.mean_fitness:.4f} | best_ever={best_fit:.4f}")

# Besten Zustand anwenden und speichern
es_selector.set_flat_params(best_params)
es_model.save_pretrained(ES_OUT / "model")
es_tokenizer.save_pretrained(ES_OUT / "model")

with open(ES_OUT / "es_history.json", "w", encoding="utf-8") as f:
    json.dump(es_history, f, indent=2)

print("ES model saved to:", ES_OUT / "model")